# Russian geography domain — `geo_ru`

Fast, diverse Wikidata generator for Russian geography multi-hop queries (cities, rivers, lakes, mountains, protected areas, and administrative divisions).

**Produces:**
- `out_wikidata_benchmark/domain_outputs/geo_ru.jsonl` — 55 benchmark examples across L1–L5


## 1. Load common helpers

The domain notebook can be run either after `00_common_helpers.ipynb` or standalone next to `common_helpers.py`.


In [1]:
from pathlib import Path

if "BenchmarkExample" not in globals():
    helper_path = Path("common_helpers.py")
    if not helper_path.exists():
        helper_path = Path("/mnt/data/common_helpers.py")
    exec(helper_path.read_text(encoding="utf-8"), globals())


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Target plan is fixed at 55 examples and skewed toward hard multi-hop levels.


In [2]:
from __future__ import annotations

import json
import random
import re
import time
from collections import Counter, defaultdict
from dataclasses import asdict
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple

try:
    from tqdm.auto import tqdm
except Exception:  # pragma: no cover - tqdm is optional in batch runs
    tqdm = None

GEO_RU_SEED = 20260521
GEO_RU_RNG = random.Random(GEO_RU_SEED)

# Fixed target plan requested for the final geo_ru add-on domain: 55 examples total.
# L1–L2: 13 simple/moderate; L3–L5: 42 hard multi-hop examples.
GEO_RU_TARGET_PLAN: Dict[str, int] = {
    "L1": 5,
    "L2": 8,
    "L3": 12,
    "L4": 15,
    "L5": 15,
}

GEO_RU_REQUESTED_BY_LEVEL = {"L1": 5, "L2": 5, "L3": 4, "L4": 3, "L5": 3}

# Force comparator diversity in the accepted dataset.  Earlier versions chose
# numeric modes randomly, so a short prefix could accidentally contain only
# "at least / не менее" examples.  The dataset loop uses this plan by accepted
# position inside each complexity level.  If one mode is impossible for a
# specific random template/anchor, the generator retries with other templates
# before falling back.
GEO_RU_NUMERIC_MODE_PLAN: Dict[str, Tuple[str, ...]] = {
    # Comparator plan is aligned with the template plan below.  We keep min/max/range
    # diversity, but avoid back-to-back near-duplicate river/admin-centre tasks.
    "L1": ("min", "min", "range", "max", "range"),
    "L2": ("max", "range", "max", "min", "min", "range", "range", "max"),
    "L3": ("range", "max", "range", "min", "min", "max", "range", "max", "min", "range", "max", "range"),
    "L4": ("range", "max", "min", "range", "range", "max", "min", "range", "max", "min", "range", "max", "range", "min", "max"),
    "L5": ("range", "max", "range", "min", "max", "range", "min", "range", "max", "range", "min", "max", "range", "max", "min"),
}

# Fast diversity plan.  The dataset loop asks a planned template first.  This
# avoids v11's slow behaviour where an expensive WDQS query was executed and
# only then rejected as a near-duplicate.  If the planned template fails under
# the requested numeric mode, the one-example generator can still fall back to
# other templates for robustness.
GEO_RU_TEMPLATE_PLAN_BY_LEVEL: Dict[str, Tuple[str, ...]] = {
    # v19 diversity plan: remove pure admin-centre prompts from the schedule and
    # make L3-L5 nature-heavy. We still keep a few city/human-geography prompts,
    # but the dataset should not turn into administrative-centre questions.
    "L1": (
        "geo_ru_lake_region_area",
        "geo_ru_city_region_population",
        "geo_ru_river_country_length",
        "geo_ru_lake_region_area",
        "geo_ru_city_region_population",
    ),
    "L2": (
        "geo_ru_noncapital_city_region_population",
        "geo_ru_lake_region_area",
        "geo_ru_river_country_length",
        "geo_ru_city_region_population",
        "geo_ru_lake_region_area",
        "geo_ru_river_country_length",
        "geo_ru_noncapital_city_region_population",
        "geo_ru_lake_region_area",
    ),
    "L3": (
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_protected_areas_subject_capital",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
    ),
    "L4": (
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_protected_area_same_subject_high_mountain",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_cities_on_tributaries",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_city_same_subject_as_large_lake",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_protected_area_same_subject_high_mountain",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_cities_on_tributaries",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_protected_area_same_subject_high_mountain",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
    ),
    "L5": (
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_cities_on_tributaries",
        "geo_ru_protected_area_same_subject_high_mountain",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_city_same_subject_as_large_lake",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_cities_on_tributaries",
        "geo_ru_protected_area_same_subject_high_mountain",
        "geo_ru_mountain_or_volcano_region_elevation",
        "geo_ru_protected_areas_region_inception",
        "geo_ru_rivers_mouth_to_sea_length",
        "geo_ru_protected_area_same_subject_high_mountain",
        "geo_ru_cities_on_tributaries",
        "geo_ru_protected_areas_region_inception",
    ),
}

# Pure admin-centre templates are intentionally disabled in the final balanced
# geo_ru notebook. The functions remain in the file for compatibility, but the
# scheduler and fallback planner must not choose them.
GEO_RU_DISABLED_TEMPLATE_IDS: set[str] = {
    "geo_ru_admin_center_subject_population",
    "geo_ru_admin_centers_subject_area",
}

_GEO_RU_FORCED_NUMERIC_MODE: Optional[str] = None
# Quality gates for accepted examples.  We require at least the requested
# number of answers and skip broad queries where the fully-labelled gold set
# exceeds 100 items.  The SELECT query uses LIMIT 101 to prove the “<=100” gate.
GEO_RU_MIN_GOLD_BY_LEVEL = dict(GEO_RU_REQUESTED_BY_LEVEL)
GEO_RU_MAX_ACCEPTED_GOLD = globals().get("GEO_RU_MAX_ACCEPTED_GOLD", 100)
GEO_RU_PROBE_LIMIT = GEO_RU_MAX_ACCEPTED_GOLD + 1
GEO_RU_GOLD_LIMIT = globals().get("GEO_RU_GOLD_LIMIT", GEO_RU_MAX_ACCEPTED_GOLD)

# For final dataset generation prefer fresh WDQS calls over old local cache.
# Set True manually only for debugging/re-running identical queries.
GEO_RU_USE_WDQS_CACHE = globals().get("GEO_RU_USE_WDQS_CACHE", False)
# In-run cache only: never reads old disk cache when GEO_RU_USE_WDQS_CACHE=False,
# but avoids paying twice for the exact same WDQS query during retries/restarts
# inside the same kernel. This is safe for one dataset build and helps speed.
GEO_RU_USE_MEMORY_WDQS_CACHE = globals().get("GEO_RU_USE_MEMORY_WDQS_CACHE", True)
_GEO_RU_MEMORY_WDQS_ROWS_CACHE: Dict[str, List[Dict[str, str]]] = globals().get("_GEO_RU_MEMORY_WDQS_ROWS_CACHE", {})

# Generation speed guard: WDQS occasionally spends 30+ seconds on unpromising
# candidate queries.  During geo_ru generation we fail such candidates fast and
# move on to another pre-planned query instead of waiting through long retry
# chains.  This does not change accepted golds; it only shortens failed probes.
GEO_RU_WDQS_FAIL_FAST = globals().get("GEO_RU_WDQS_FAIL_FAST", True)
GEO_RU_WDQS_FAST_TIMEOUT_SECONDS = int(globals().get("GEO_RU_WDQS_FAST_TIMEOUT_SECONDS", 14))
GEO_RU_WDQS_FAST_MAX_RETRIES = int(globals().get("GEO_RU_WDQS_FAST_MAX_RETRIES", 1))
GEO_RU_GENERATOR_MAX_ATTEMPTS = int(globals().get("GEO_RU_GENERATOR_MAX_ATTEMPTS", 3))

# Labels policy: the answer entity must have both labels; EN must be Latin-script.
GEO_RU_REQUIRE_BOTH_RU_EN_LABELS = True
GEO_RU_REQUIRE_LATIN_EN_LABELS = True

# Keep QID gold canonical. Duplicate public labels are noted in metadata,
# but they are not a hard skip by default: for Russian geography many distinct
# lakes/settlements legitimately share short transliterated labels, and hard
# rejection makes max/range generation very slow.
GEO_RU_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS = globals().get("GEO_RU_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS", False)

GEO_RU_DOMAIN_OUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
GEO_RU_DOMAIN_OUT_DIR.mkdir(parents=True, exist_ok=True)
GEO_RU_OUTPUT_PATH = GEO_RU_DOMAIN_OUT_DIR / "geo_ru.jsonl"
GEO_RU_AUDIT_PATH = GEO_RU_DOMAIN_OUT_DIR / "geo_ru_generation_audit.json"
GEO_RU_CHECKPOINT_PATH = GEO_RU_DOMAIN_OUT_DIR / "geo_ru_generation_checkpoint.json"

# Set to False if you only want to import/register the generator.
RUN_GEO_RU_GENERATION = globals().get("RUN_GEO_RU_GENERATION", True)
OVERWRITE_GEO_RU_OUTPUT = globals().get("OVERWRITE_GEO_RU_OUTPUT", True)


## 3. Generic helpers

Small NLG, SPARQL, label, quantity-unit, and record-key helpers used by all templates.

Important quality rules live here: SELECT requires both `ru` and `en` labels; `en` labels with Cyrillic fallback text are rejected; WDQS is called with `GEO_RU_USE_WDQS_CACHE=False` by default for final generation.


In [3]:
_QID_RE_GEO_RU = re.compile(r"^Q\d+$")
_CYRILLIC_RE_GEO_RU = re.compile(r"[\u0400-\u052F]")


def _gr_escape(s: str) -> str:
    return str(s).replace("\\", "\\\\").replace('"', '\\"')


def _gr_norm_key(s: Any) -> str:
    return re.sub(r"\s+", " ", str(s or "").strip()).casefold()


def _gr_fmt_int(n: int) -> str:
    return f"{int(n):,}".replace(",", " ")


def _gr_fmt_people_ru(n: int) -> str:
    n = int(n)
    if n >= 1_000_000:
        v = n / 1_000_000
        if abs(v - round(v)) < 1e-9:
            return f"{int(round(v))} млн"
        return f"{v:.1f}".replace(".", ",") + " млн"
    if n % 1000 == 0:
        return f"{n // 1000} тысяч"
    return _gr_fmt_int(n)


def _gr_ru_count(n: int, one: str, few: str, many: str) -> str:
    n_abs = abs(int(n))
    n100, n10 = n_abs % 100, n_abs % 10
    if 11 <= n100 <= 14:
        return many
    if n10 == 1:
        return one
    if 2 <= n10 <= 4:
        return few
    return many


def _gr_entity(qid: str, label_en: str) -> Dict[str, str]:
    return {"qid": qid, "label_en": label_en}


def _gr_clean_constraints(obj: Any) -> Any:
    """Drop empty values and keep constraints JSON clean and English-only."""
    if isinstance(obj, dict):
        cleaned = {}
        for k, v in obj.items():
            if v is None or v == "" or v == [] or v == {}:
                continue
            cleaned[k] = _gr_clean_constraints(v)
        return cleaned
    if isinstance(obj, list):
        return [_gr_clean_constraints(x) for x in obj if x is not None]
    return obj


# These keys are technical metadata, not public constraints.  Public constraints
# must describe only the semantic criteria present in query_text, in the same
# flat style as the existing final JSONL domains (cinema / geo_international):
# strings, numbers and booleans; no QIDs, no label/QID dicts, no validation flags.
_GEO_RU_FORBIDDEN_PUBLIC_CONSTRAINT_KEYS = {
    "template_id",
    "template_family",
    "constraint_language",
    "constraints_are_wdqs_only",
    "requires_ru_label",
    "requires_en_label",
    "requires_latin_script_en_label",
    "max_gold_allowed",
    "skip_if_rows_returned_reach_limit",
    "skip_if_gold_count_gt",
    "wdqs_candidate_limit",
    "gold_limit",
    "excluded_answer_types",
    "same_administrative_entity_lake_excluded_types",
    "country_statement_interpreted_as",
    "population_unit",
}

_GEO_RU_PUBLIC_CONSTRAINT_KEY_RENAMES = {
    # Match geo_international/cinema style: answer class is stored as `kind`.
    "answer_type": "kind",
    # Keep public constraints compact and readable; QIDs stay in SPARQL/meta.
    "located_in_administrative_entity": "administrative_entity",
    "area_min_km2": "area_min_sqkm",
    "area_max_km2": "area_max_sqkm",
    "federal_subject_area_min_km2": "federal_subject_area_min_sqkm",
    "federal_subject_area_max_km2": "federal_subject_area_max_sqkm",
    "same_administrative_entity_has_lake_area_min_km2": "same_administrative_entity_has_lake_area_min_sqkm",
    "same_administrative_entity_has_lake_area_max_km2": "same_administrative_entity_has_lake_area_max_sqkm",
    "lake_area_min_km2": "lake_area_min_sqkm",
    "lake_area_max_km2": "lake_area_max_sqkm",
    "located_in_federal_subject_whose_capital_population_min": "federal_subject_capital_population_min",
    "mouth_of_watercourse_one_of": "mouth_body_one_of",
}


def _gr_constraint_public_value(v: Any) -> Any:
    """Convert implementation values into public JSONL constraint values.

    Examples:
      {"qid": "Q159", "label_en": "Russia"} -> "Russia"
      [{"qid": "Q166", "label_en": "Black Sea"}, ...] -> ["Black Sea", ...]
    """
    if isinstance(v, dict):
        if "label_en" in v and "qid" in v:
            return str(v["label_en"])
        out: Dict[str, Any] = {}
        for kk, vv in v.items():
            if kk in _GEO_RU_FORBIDDEN_PUBLIC_CONSTRAINT_KEYS:
                continue
            public_key = _GEO_RU_PUBLIC_CONSTRAINT_KEY_RENAMES.get(kk, kk)
            public_val = _gr_constraint_public_value(vv)
            if public_val is None or public_val == "" or public_val == [] or public_val == {}:
                continue
            out[public_key] = public_val
        return out
    if isinstance(v, list):
        out_list = []
        for x in v:
            public_x = _gr_constraint_public_value(x)
            if public_x is None or public_x == "" or public_x == [] or public_x == {}:
                continue
            out_list.append(public_x)
        return out_list
    return v


def _gr_public_constraints(obj: Any) -> Any:
    """Return clean, user-facing constraints only.

    The output intentionally mirrors the already-finalized domains:
    - shallow semantic keys such as `kind`, `country`, `continent`,
      `area_min_sqkm`, `length_min_km`;
    - plain strings/numbers/bools/lists;
    - no Wikidata QIDs, no template IDs, no label-policy/limit metadata.
    """
    public = _gr_constraint_public_value(_gr_clean_constraints(obj))
    if isinstance(public, dict):
        return {k: v for k, v in public.items() if k not in _GEO_RU_FORBIDDEN_PUBLIC_CONSTRAINT_KEYS}
    return public


def _gr_assert_public_constraints_clean(obj: Any) -> None:
    """Fail fast if public constraints leak implementation details."""
    def walk(x: Any, path: str = "constraints") -> None:
        if isinstance(x, dict):
            assert "qid" not in x, f"QID leaked into public constraints at {path}"
            assert "label_en" not in x, f"label_en dict leaked into public constraints at {path}"
            for kk, vv in x.items():
                assert kk not in _GEO_RU_FORBIDDEN_PUBLIC_CONSTRAINT_KEYS, f"technical key leaked into public constraints: {path}.{kk}"
                assert kk not in _GEO_RU_PUBLIC_CONSTRAINT_KEY_RENAMES, f"unrenamed key leaked into public constraints: {path}.{kk}"
                walk(vv, f"{path}.{kk}")
        elif isinstance(x, list):
            for j, xx in enumerate(x):
                walk(xx, f"{path}[{j}]")
    walk(obj)

def _gr_type_line(var: str, class_qid: str) -> str:
    return f"?{var} wdt:P31/wdt:P279* wd:{class_qid} ."


def _gr_federal_subject_lines(var: str = "subject") -> List[str]:
    """SPARQL lines for all Russian federal-subject entity types.

    Important: Q835714 is only "oblast of Russia"; federal subjects also include
    republics, krais, federal cities, autonomous oblast and autonomous okrugs.
    """
    values = " ".join(f"wd:{qid}" for qid in Q_RUSSIAN_FEDERAL_SUBJECT_TYPES)
    return [
        f"VALUES ?{var}_type {{ {values} }}",
        f"?{var} wdt:P31 ?{var}_type .",
    ]


def _gr_country_russia_line(var: str) -> str:
    return f"?{var} wdt:P17 wd:{Q_RUSSIA} ."


def _gr_values_qids(var: str, qids: Sequence[str]) -> str:
    vals = " ".join(f"wd:{q}" for q in dict.fromkeys(qids) if _QID_RE_GEO_RU.fullmatch(str(q)))
    return f"VALUES ?{var} {{ {vals} }}"


def _gr_label_is_noise(label: str) -> bool:
    s = str(label or "").strip()
    if not s:
        return True
    if _QID_RE_GEO_RU.fullmatch(s):
        return True
    if len(s) > 160:
        return True
    return False


def _gr_en_label_is_bad(label: str) -> bool:
    """Reject missing/QID labels and Cyrillic fallbacks masquerading as English."""
    if _gr_label_is_noise(label):
        return True
    if GEO_RU_REQUIRE_LATIN_EN_LABELS and _CYRILLIC_RE_GEO_RU.search(str(label)):
        return True
    return False


# Unit QIDs used by Wikidata quantities.
Q_UNIT_METRE = "Q11573"
Q_UNIT_KILOMETRE = "Q828224"
Q_UNIT_SQUARE_METRE = "Q25343"
Q_UNIT_SQUARE_KILOMETRE = "Q712226"
Q_UNIT_HECTARE = "Q35852"


def _gr_quantity_lines(item_var: str, prop: str, out_var: str, unit_kind: str) -> List[str]:
    """Return robust normalized quantity lines for WDQS p:/psv: statements.

    unit_kind:
      - 'length_km' normalizes metres to kilometres;
      - 'elevation_m' normalizes kilometres to metres;
      - 'area_km2' normalizes square metres and hectares to square kilometres.
    """
    stmt = f"{out_var}_stmt"
    node = f"{out_var}_node"
    raw = f"{out_var}_raw"
    unit = f"{out_var}_unit"
    lines = [
        f"?{item_var} p:{prop} ?{stmt} .",
        f"?{stmt} psv:{prop} ?{node} .",
        f"?{node} wikibase:quantityAmount ?{raw} .",
        f"?{node} wikibase:quantityUnit ?{unit} .",
    ]
    if unit_kind == "length_km":
        lines.append(
            f"BIND(IF(?{unit} = wd:{Q_UNIT_METRE}, ?{raw} / 1000, ?{raw}) AS ?{out_var}) ."
        )
    elif unit_kind == "elevation_m":
        lines.append(
            f"BIND(IF(?{unit} = wd:{Q_UNIT_KILOMETRE}, ?{raw} * 1000, ?{raw}) AS ?{out_var}) ."
        )
    elif unit_kind == "area_km2":
        lines.append(
            "BIND("
            f"IF(?{unit} = wd:{Q_UNIT_SQUARE_METRE}, ?{raw} / 1000000, "
            f"IF(?{unit} = wd:{Q_UNIT_HECTARE}, ?{raw} / 100, ?{raw}))"
            f" AS ?{out_var}) ."
        )
    else:
        lines.append(f"BIND(?{raw} AS ?{out_var}) .")
    return lines


def _gr_year_max_line(date_var: str, year_max: int) -> str:
    return f"FILTER(YEAR(?{date_var}) <= {int(year_max)}) ."


def _gr_year_min_line(date_var: str, year_min: int) -> str:
    return f"FILTER(YEAR(?{date_var}) >= {int(year_min)}) ."


def _gr_strict_label_lines(var: str, require_latin_en: bool = True) -> List[str]:
    lines = [
        f'?{var} rdfs:label ?{var}LabelRu FILTER(LANG(?{var}LabelRu) = "ru") .',
        f'?{var} rdfs:label ?{var}LabelEn FILTER(LANG(?{var}LabelEn) = "en") .',
    ]
    if require_latin_en:
        # Reject accidental Russian fallback strings stored/returned as English labels.
        lines.append(f'FILTER(!REGEX(STR(?{var}LabelEn), "[А-Яа-яЁё]")) .')
    return lines


def _gr_build_select_query(where_lines: Sequence[str], item_var: str = "item", limit: int = 100) -> str:
    where = "\n      ".join(str(x).strip() for x in where_lines if str(x).strip())
    label_lines = "\n      ".join(_gr_strict_label_lines(item_var, require_latin_en=GEO_RU_REQUIRE_LATIN_EN_LABELS))
    return f"""
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu ?{item_var}LabelEn WHERE {{
      {where}
      {label_lines}
    }}
    ORDER BY LCASE(STR(?{item_var}LabelEn))
    LIMIT {int(limit)}
    """.strip()


def _gr_build_ask_query(where_lines: Sequence[str], item_var: str = "item") -> str:
    where = "\n      ".join(str(x).strip() for x in where_lines if str(x).strip())
    return f"""
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      {where}
    }}
    """.strip()


def _gr_rows_from_wdqs(query: str) -> List[Dict[str, str]]:
    """Run WDQS and return rows.

    Disk cache stays disabled by default for final freshness.  We still use a
    run-local memory cache and fail-fast timeout/retry settings so bad candidate
    probes do not stall the whole notebook.
    """
    q = query.strip()
    if bool(globals().get("GEO_RU_USE_MEMORY_WDQS_CACHE", True)):
        mem = globals().setdefault("_GEO_RU_MEMORY_WDQS_ROWS_CACHE", {})
        if q in mem:
            return list(mem[q])

    wd_obj = globals().get("wd", None)
    old_timeout = getattr(wd_obj, "timeout", None) if wd_obj is not None else None
    old_retries = getattr(wd_obj, "max_retries", None) if wd_obj is not None else None
    try:
        if bool(globals().get("GEO_RU_WDQS_FAIL_FAST", True)) and wd_obj is not None:
            try:
                wd_obj.timeout = int(globals().get("GEO_RU_WDQS_FAST_TIMEOUT_SECONDS", 14))
                wd_obj.max_retries = int(globals().get("GEO_RU_WDQS_FAST_MAX_RETRIES", 1))
            except Exception:
                pass

        data = wd.sparql_select(q, use_cache=bool(GEO_RU_USE_WDQS_CACHE))
        rows = rows_from_select(data)
        if bool(globals().get("GEO_RU_USE_MEMORY_WDQS_CACHE", True)):
            globals().setdefault("_GEO_RU_MEMORY_WDQS_ROWS_CACHE", {})[q] = list(rows)
        return rows
    finally:
        if wd_obj is not None:
            try:
                if old_timeout is not None:
                    wd_obj.timeout = old_timeout
                if old_retries is not None:
                    wd_obj.max_retries = old_retries
            except Exception:
                pass


def _gr_collect_gold(where_lines: Sequence[str], item_var: str, wdqs_limit: int) -> Tuple[str, List[Dict[str, str]], Dict[str, int]]:
    query = _gr_build_select_query(where_lines=where_lines, item_var=item_var, limit=wdqs_limit)
    rows = _gr_rows_from_wdqs(query)
    gold: List[Dict[str, str]] = []
    seen: set[str] = set()
    label_sources = Counter()

    for r in rows:
        qid = uri_to_qid(r.get(item_var, ""))
        ru = (r.get(f"{item_var}LabelRu") or "").strip()
        en = (r.get(f"{item_var}LabelEn") or "").strip()
        if not qid or qid in seen:
            continue
        if _gr_label_is_noise(ru) or _gr_en_label_is_bad(en):
            continue
        seen.add(qid)
        label_sources["ru_label"] += 1
        label_sources["en_label"] += 1
        gold.append({"qid": qid, "label_ru": ru, "label_en": en})

    return query, gold, dict(label_sources)


def _gr_finalize_text(q_ru: str, q_en: str, total_gold: int, requested_count: int) -> Tuple[str, str]:
    """Return the natural-language query exactly as authored by the template.

    We intentionally do not append instructions such as "list any of them" or
    "list all matching answers": requested_count already defines how many
    answers the benchmark asks for, and broad/full-gold metadata is stored in
    separate JSON fields.
    """
    return q_ru.rstrip(), q_en.rstrip()


def _gr_record_key(ex: BenchmarkExample) -> Tuple[str, str, str]:
    """Dataset-level duplicate key.

    Keep it independent of the requested count and complexity level: the same
    semantic task should not appear twice as L3/L4 only because the prompt asks
    for 4 versus 3 examples.  This is checked after a candidate is collected,
    but it prevents exact duplicate records from entering the final JSONL while
    keeping the generator fast.
    """
    constraints = getattr(ex, "constraints", {}) or {}
    stable = json.dumps(constraints, ensure_ascii=False, sort_keys=True)
    return (ex.template_id or "", _gr_norm_key(stable), "")


## 4. Domain vocabulary

QIDs are kept explicit for reproducibility. All structured constraints use English labels and QIDs; Russian strings are used only for Russian natural-language questions.


In [4]:
# Core classes/entities.
Q_RUSSIA = "Q159"
Q_CITY = "Q515"
Q_VILLAGE = "Q532"
Q_RIVER = "Q4022"
Q_LAKE = "Q23397"
Q_RESERVOIR = "Q131681"
Q_WATER_RESERVOIR = "Q11727010"
Q_MOUNTAIN = "Q8502"
Q_VOLCANO = "Q8072"
Q_PROTECTED_AREA = "Q473972"
# Russian federal subjects are not a subclass tree under one Wikidata class:
# Q835714 is only "oblast of Russia".  Use all current subject-type classes.
Q_OBLAST_RUSSIA = "Q835714"
Q_REPUBLIC_RUSSIA = "Q41162"
Q_FEDERAL_CITY_RUSSIA = "Q183342"
Q_KRAI_RUSSIA = "Q831740"
Q_AUTONOMOUS_OBLAST_RUSSIA = "Q309166"
Q_AUTONOMOUS_OKRUG_RUSSIA = "Q184122"
Q_FEDERAL_SUBJECT_RUSSIA = Q_OBLAST_RUSSIA  # backward-compatible alias; do not use alone in SPARQL
Q_RUSSIAN_FEDERAL_SUBJECT_TYPES = [
    Q_OBLAST_RUSSIA,
    Q_REPUBLIC_RUSSIA,
    Q_FEDERAL_CITY_RUSSIA,
    Q_KRAI_RUSSIA,
    Q_AUTONOMOUS_OBLAST_RUSSIA,
    Q_AUTONOMOUS_OKRUG_RUSSIA,
]

COUNTRY_RUSSIA = _gr_entity(Q_RUSSIA, "Russia")



def _gr_strict_city_lines(var: str) -> List[str]:
    """City constraints that reject settlement/village leakage from Wikidata.

    Some Russian locality items are reachable through the broad Q515 subclass
    path but have labels like "village" / "settlement".  Public prompts that
    say "city" should not accept those items, so city templates use this helper
    instead of a bare `_gr_type_line(..., Q_CITY)`.
    """
    return [
        _gr_type_line(var, Q_CITY),
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_VILLAGE} . }}",
        f'?{var} rdfs:label ?{var}CityLabelRu FILTER(LANG(?{var}CityLabelRu) = "ru") .',
        f'?{var} rdfs:label ?{var}CityLabelEn FILTER(LANG(?{var}CityLabelEn) = "en") .',
        f'FILTER(!REGEX(LCASE(STR(?{var}CityLabelEn)), "village|settlement|hamlet|rural locality|selo|stanitsa")) .',
        f'FILTER(!REGEX(LCASE(STR(?{var}CityLabelRu)), "деревня|село|пос[её]лок|хутор|станица|аул")) .',
        f'FILTER(!REGEX(STR(?{var}CityLabelEn), "[А-Яа-яЁё]")) .',
    ]


def _gr_natural_lake_lines(var: str) -> List[str]:
    """Lake constraints that reject reservoirs/artificial storage lakes.

    Wikidata models reservoirs as subclasses of lake/artificial lake in many places;
    for prompts saying “lake”, exclude explicit reservoir classes and label patterns.
    """
    return [
        _gr_type_line(var, Q_LAKE),
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_RESERVOIR} . }}",
        f"MINUS {{ ?{var} wdt:P31/wdt:P279* wd:{Q_WATER_RESERVOIR} . }}",
        f'?{var} rdfs:label ?{var}NaturalLakeLabelRu FILTER(LANG(?{var}NaturalLakeLabelRu) = "ru") .',
        f'?{var} rdfs:label ?{var}NaturalLakeLabelEn FILTER(LANG(?{var}NaturalLakeLabelEn) = "en") .',
        f'FILTER(!REGEX(LCASE(STR(?{var}NaturalLakeLabelEn)), "reservoir|impoundment|dam lake|storage lake")) .',
        f'FILTER(!REGEX(LCASE(STR(?{var}NaturalLakeLabelRu)), "водохранилище|водохр")) .',
        f'FILTER(!REGEX(STR(?{var}NaturalLakeLabelEn), "[А-Яа-яЁё]")) .',
    ]


REGION_SPECS: List[Dict[str, Any]] = [
    {"qid": "Q1697", "label_en": "Moscow Oblast", "ru_prep": "Московской области", "city_pop_min": [30_000, 50_000, 100_000], "lake_area_min": [3, 5]},
    {"qid": "Q2191", "label_en": "Leningrad Oblast", "ru_prep": "Ленинградской области", "city_pop_min": [20_000, 40_000, 70_000], "lake_area_min": [5, 10]},
    {"qid": "Q3680", "label_en": "Krasnodar Krai", "ru_prep": "Краснодарском крае", "city_pop_min": [30_000, 70_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q5481", "label_en": "Tatarstan", "ru_prep": "Республике Татарстан", "city_pop_min": [30_000, 60_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q5710", "label_en": "Bashkortostan", "ru_prep": "Республике Башкортостан", "city_pop_min": [25_000, 50_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q5462", "label_en": "Sverdlovsk Oblast", "ru_prep": "Свердловской области", "city_pop_min": [30_000, 60_000, 100_000], "lake_area_min": [3, 5]},
    {"qid": "Q5851", "label_en": "Novosibirsk Oblast", "ru_prep": "Новосибирской области", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [10, 20]},
    {"qid": "Q5400", "label_en": "Perm Krai", "ru_prep": "Пермском крае", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q4341", "label_en": "Primorsky Krai", "ru_prep": "Приморском крае", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q1914", "label_en": "Karelia", "ru_prep": "Республике Карелия", "city_pop_min": [10_000, 20_000, 50_000], "lake_area_min": [20, 50, 100]},
    {"qid": "Q7948", "label_en": "Kamchatka Krai", "ru_prep": "Камчатском крае", "city_pop_min": [5_000, 10_000, 30_000], "lake_area_min": [1, 3, 5]},
    {"qid": "Q5971", "label_en": "Altai Republic", "ru_prep": "Республике Алтай", "city_pop_min": [5_000, 10_000], "lake_area_min": [1, 3]},
    {"qid": "Q5267", "label_en": "Kabardino-Balkaria", "ru_prep": "Кабардино-Балкарии", "city_pop_min": [10_000, 20_000, 50_000], "lake_area_min": [1, 3]},
    {"qid": "Q5328", "label_en": "Karachay-Cherkessia", "ru_prep": "Карачаево-Черкесии", "city_pop_min": [10_000, 20_000, 50_000], "lake_area_min": [1, 3]},
    {"qid": "Q6563", "label_en": "Krasnoyarsk Krai", "ru_prep": "Красноярском крае", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [10, 50, 100]},
    {"qid": "Q6585", "label_en": "Irkutsk Oblast", "ru_prep": "Иркутской области", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [50, 100]},
    {"qid": "Q3573", "label_en": "Rostov Oblast", "ru_prep": "Ростовской области", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q3819", "label_en": "Volgograd Oblast", "ru_prep": "Волгоградской области", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [1, 3]},
    {"qid": "Q2246", "label_en": "Nizhny Novgorod Oblast", "ru_prep": "Нижегородской области", "city_pop_min": [20_000, 50_000, 100_000], "lake_area_min": [1, 3]},
]

RIVER_SEEDS: List[Dict[str, Any]] = [
    {"qid": "Q626", "label_en": "Volga", "ru_prep": "Волге", "ru_dat": "Волгу"},
    {"qid": "Q973", "label_en": "Ob", "ru_prep": "Оби", "ru_dat": "Обь"},
    {"qid": "Q78707", "label_en": "Yenisei", "ru_prep": "Енисее", "ru_dat": "Енисей"},
    {"qid": "Q7884", "label_en": "Lena", "ru_prep": "Лене", "ru_dat": "Лену"},
    {"qid": "Q1223", "label_en": "Don", "ru_prep": "Дону", "ru_dat": "Дон"},
]

SEA_GROUPS: List[Dict[str, Any]] = [
    {
        "entities": [
            {"qid": "Q166", "label_en": "Black Sea"},
            {"qid": "Q5484", "label_en": "Caspian Sea"},
            {"qid": "Q545", "label_en": "Baltic Sea"},
        ],
        "label_en": "the Black Sea, Caspian Sea, or Baltic Sea",
        "ru": "Чёрное, Каспийское или Балтийское море",
    },
    {
        "entities": [
            {"qid": "Q45823", "label_en": "Barents Sea"},
            {"qid": "Q16460", "label_en": "Sea of Okhotsk"},
            {"qid": "Q16004", "label_en": "Kara Sea"},
        ],
        "label_en": "the Barents Sea, Sea of Okhotsk, or Kara Sea",
        "ru": "Баренцево, Охотское или Карское море",
    },
]

ELEVATED_REGION_SPECS = [r for r in REGION_SPECS if r["qid"] in {"Q7948", "Q5971", "Q5267", "Q5328", "Q6563"}]


## 5. Finalization and validation

One finalizer is shared by all templates. It runs WDQS, requires enough answers for the requested count, skips candidates with more than 100 fully-labelled gold answers, and refuses any example that could be truncated by a WDQS/local limit.


In [5]:
def _gr_duplicate_gold_labels(gold: Sequence[Dict[str, str]]) -> Dict[str, List[str]]:
    """Return duplicate public answer labels that would make label-level gold ambiguous."""
    out: Dict[str, List[str]] = {}
    for field in ("label_ru", "label_en"):
        counts = Counter(_gr_norm_key(g.get(field, "")) for g in gold if g.get(field))
        dups = sorted({g.get(field, "").strip() for g in gold if counts.get(_gr_norm_key(g.get(field, "")), 0) > 1})
        if dups:
            out[field] = dups
    return out


def _gr_finalize_wdqs_example(
    *,
    idx: int,
    complexity: str,
    template_id: str,
    template_family: str,
    q_ru: str,
    q_en: str,
    constraints: Dict[str, Any],
    where_lines: Sequence[str],
    requested_count: Optional[int] = None,
    item_var: str = "item",
    wdqs_limit: Optional[int] = None,
    gold_limit: Optional[int] = None,
) -> Optional[BenchmarkExample]:
    requested_count = int(requested_count or GEO_RU_REQUESTED_BY_LEVEL.get(complexity, 3))
    min_gold = max(int(GEO_RU_MIN_GOLD_BY_LEVEL.get(complexity, requested_count)), requested_count)
    max_accepted_gold = int(GEO_RU_MAX_ACCEPTED_GOLD)

    # Always probe with max+1.  If WDQS returns the extra row, the real gold set
    # is too broad (> max_accepted_gold) or potentially incomplete under LIMIT.
    wdqs_limit = max_accepted_gold + 1
    gold_limit = min(int(gold_limit or GEO_RU_GOLD_LIMIT), int(GEO_RU_GOLD_LIMIT), max_accepted_gold)

    try:
        sparql_query, gold_all, label_sources = _gr_collect_gold(where_lines, item_var=item_var, wdqs_limit=wdqs_limit)
    except Exception as e:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[geo_ru][{template_id}] WDQS failed: {type(e).__name__}: {e}")
        return None

    total_before_limit = len(gold_all)

    # Hard quality gates:
    # 1) enough answers for the requested count;
    # 2) no broad examples where gold exceeds 100;
    # 3) no accepted example may be truncated by WDQS/local limits.
    if total_before_limit < min_gold:
        return None
    if total_before_limit > max_accepted_gold or total_before_limit >= wdqs_limit:
        return None
    if total_before_limit > gold_limit:
        return None

    # Duplicate public labels are common in geography (for example several
    # distinct lakes named "Gornoe"). QIDs remain the canonical gold answers,
    # so by default we keep such examples and record the ambiguity in metadata.
    # Set GEO_RU_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS=True only for a stricter,
    # much slower debugging run.
    duplicate_labels = _gr_duplicate_gold_labels(gold_all)
    if duplicate_labels and GEO_RU_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS:
        if globals().get("DEBUG_GENERATOR_ERRORS", False):
            print(f"[geo_ru][{template_id}] skipped: duplicate gold labels {duplicate_labels}")
        return None

    gold = gold_all
    total_gold = len(gold)
    q_ru_final, q_en_final = _gr_finalize_text(q_ru, q_en, total_gold, requested_count)
    gold_truncated = False

    # Keep public constraints clean and semantic: only the criteria explicitly
    # present in the user-facing query. Technical validation/gold-collection
    # details live in gold_collection_meta.
    clean_constraints = _gr_public_constraints(constraints)
    _gr_assert_public_constraints_clean(clean_constraints)
    meta = {
        "source": "wikidata_sparql",
        "constraints_are_wdqs_only": True,
        "wdqs_candidate_limit": wdqs_limit,
        "max_gold_allowed": max_accepted_gold,
        "skip_if_rows_returned_reach_limit": True,
        "skip_if_gold_count_gt": max_accepted_gold,
        "requires_both_ru_and_en_labels": True,
        "requires_latin_script_en_label": bool(GEO_RU_REQUIRE_LATIN_EN_LABELS),
        "requires_unique_gold_labels": bool(GEO_RU_ENFORCE_UNIQUE_PUBLIC_GOLD_LABELS),
        "duplicate_public_gold_labels": duplicate_labels,
        "rows_returned_by_wdqs_after_label_filter": total_before_limit,
        "gold_limit": gold_limit,
        "gold_returned": total_gold,
        "gold_total_before_limit": total_before_limit,
        "gold_may_be_incomplete_due_to_wdqs_limit": False,
        "gold_truncated_by_local_limit": False,
        "label_sources": label_sources,
        "template_id": template_id,
        "template_family": template_family,
    }

    return BenchmarkExample(
        id=f"geo_ru_{complexity.lower()}_{idx:04d}",
        domain="geo_ru",
        complexity=complexity,
        query_text_ru=q_ru_final,
        query_text_en=q_en_final,
        constraints=clean_constraints,
        requested_count=requested_count,
        gold_answer_qids=[x["qid"] for x in gold],
        gold_answer_labels_ru=[x["label_ru"] for x in gold],
        gold_answer_labels_en=[x["label_en"] for x in gold],
        sparql_query=sparql_query,
        created_at=utc_now_z(),
        is_advanced=complexity in {"L3", "L4", "L5"},
        template_id=template_id,
        template_family=template_family,
        gold_truncated=gold_truncated,
        ask_validator_sparql=_gr_build_ask_query(where_lines, item_var=item_var),
        gold_collection_meta=meta,
    )


def _gr_failed_example(complexity: str, idx: int) -> BenchmarkExample:
    return BenchmarkExample(
        id=f"geo_ru_fail_{complexity.lower()}_{idx:04d}",
        domain="geo_ru",
        complexity=complexity,
        query_text_ru="(ошибка генерации geo_ru)",
        query_text_en="(geo_ru generation failed)",
        constraints={"failed": True, "complexity": complexity},
        requested_count=0,
        gold_answer_qids=[],
        gold_answer_labels_ru=[],
        gold_answer_labels_en=[],
        sparql_query="",
        created_at=utc_now_z(),
        template_id="geo_ru_failed",
        template_family="failed",
    )


## 6. L1–L2 templates

Simpler templates still use real Wikidata constraints: type, country, administrative containment, population, and normalized quantities.


In [6]:
# Numeric prompts should not all be “at least / не менее”.  These helpers let
# templates use minimum, maximum, or closed range constraints while keeping the
# public JSONL constraints clean and aligned with the natural-language query.

BoundSpec = Dict[str, Optional[int]]


def _gr_pick_bound(
    rng: random.Random,
    values: Sequence[int],
    modes: Sequence[str] = ("min", "max", "range"),
) -> BoundSpec:
    """Pick a numeric bound.

    `GEO_RU_NUMERIC_MODE_PLAN` can force the comparator mode during dataset
    generation.  Thresholds are deliberately biased toward useful acceptance:
    - min: choose a middle/high threshold so the gold set is not too broad;
    - max: choose a low/middle threshold so "at most" examples are selective;
    - range: choose a comparatively narrow interval.
    """
    vals = sorted({int(v) for v in values})
    if not vals:
        raise ValueError("empty numeric bound values")

    allowed = [m for m in modes if m in {"min", "max", "range"}]
    if len(vals) < 2:
        allowed = [m for m in allowed if m != "range"] or ["min"]
    if not allowed:
        allowed = ["min"]

    forced = globals().get("_GEO_RU_FORCED_NUMERIC_MODE")
    mode = forced if forced in allowed else rng.choice(list(allowed))

    if mode == "min":
        # Higher thresholds reduce over-broad gold sets and produce stronger
        # criteria than ">= the lowest configured number".
        pool = vals[max(0, len(vals) // 2):] or vals
        return {"min": rng.choice(pool), "max": None}

    if mode == "max":
        # Lower thresholds keep "at most" examples selective enough to pass the
        # <=100-gold gate.
        cut = max(1, (len(vals) + 1) // 2)
        pool = vals[:cut] or vals
        return {"min": None, "max": rng.choice(pool)}

    # Range: use nearby boundaries rather than the widest possible interval.
    if len(vals) == 2:
        return {"min": vals[0], "max": vals[1]}
    lo_i = rng.randrange(0, len(vals) - 1)
    hi_i_max = min(len(vals) - 1, lo_i + 2)
    hi_i = rng.randrange(lo_i + 1, hi_i_max + 1)
    return {"min": vals[lo_i], "max": vals[hi_i]}


def _gr_bound_mode(b: BoundSpec) -> str:
    if b.get("min") is not None and b.get("max") is not None:
        return "range"
    if b.get("max") is not None:
        return "max"
    if b.get("min") is not None:
        return "min"
    return "none"


def _gr_constraint_numeric_mode(constraints: Dict[str, Any]) -> str:
    """Derive the public numeric comparator mode from clean constraints."""
    if not isinstance(constraints, dict):
        return "none"
    flat = dict(constraints)
    min_keys = {k for k in flat if k.endswith("_min") or "_min_" in k or k.endswith("_min_km") or k.endswith("_min_sqkm")}
    max_keys = {k for k in flat if k.endswith("_max") or "_max_" in k or k.endswith("_max_km") or k.endswith("_max_sqkm")}
    has_min = any(k.endswith("_min") or "_min_" in k or k.endswith("_min_km") or k.endswith("_min_sqkm") for k in flat)
    has_max = any(k.endswith("_max") or "_max_" in k or k.endswith("_max_km") or k.endswith("_max_sqkm") for k in flat)
    if has_min and has_max:
        return "range_or_mixed"
    if has_max:
        return "max"
    if has_min:
        return "min"
    return "none"


def _gr_bound_filters(var_name: str, b: BoundSpec) -> List[str]:
    lines: List[str] = []
    if b.get("min") is not None:
        lines.append(f"FILTER(?{var_name} >= {int(b['min'])}) .")
    if b.get("max") is not None:
        lines.append(f"FILTER(?{var_name} <= {int(b['max'])}) .")
    return lines


def _gr_bound_constraints(b: BoundSpec, min_key: str, max_key: str) -> Dict[str, int]:
    out: Dict[str, int] = {}
    if b.get("min") is not None:
        out[min_key] = int(b["min"])
    if b.get("max") is not None:
        out[max_key] = int(b["max"])
    return out


def _gr_bound_phrase_ru(b: BoundSpec, fmt: Callable[[int], str], unit: str) -> str:
    lo, hi = b.get("min"), b.get("max")
    if lo is not None and hi is not None:
        return f"от {fmt(int(lo))} до {fmt(int(hi))} {unit}"
    if hi is not None:
        return f"не более {fmt(int(hi))} {unit}"
    return f"не менее {fmt(int(lo))} {unit}"


def _gr_bound_phrase_en(b: BoundSpec, fmt: Callable[[int], str], unit: str) -> str:
    lo, hi = b.get("min"), b.get("max")
    if lo is not None and hi is not None:
        return f"between {fmt(int(lo))} and {fmt(int(hi))} {unit}"
    if hi is not None:
        return f"at most {fmt(int(hi))} {unit}"
    return f"at least {fmt(int(lo))} {unit}"


def _gr_population_phrase_ru(b: BoundSpec) -> str:
    return _gr_bound_phrase_ru(b, _gr_fmt_people_ru, "человек")


def _gr_population_phrase_en(b: BoundSpec) -> str:
    return _gr_bound_phrase_en(b, _gr_fmt_int, "people")


def _gr_population_clause_ru(b: BoundSpec) -> str:
    return "с населением " + _gr_population_phrase_ru(b)


def _gr_population_clause_en(b: BoundSpec) -> str:
    return "with population of " + _gr_population_phrase_en(b)


def _gr_length_phrase_ru(b: BoundSpec) -> str:
    return _gr_bound_phrase_ru(b, _gr_fmt_int, "км")


def _gr_length_phrase_en(b: BoundSpec) -> str:
    return _gr_bound_phrase_en(b, _gr_fmt_int, "km")


def _gr_length_clause_ru(b: BoundSpec) -> str:
    return "общей длиной " + _gr_length_phrase_ru(b)


def _gr_length_clause_en(b: BoundSpec) -> str:
    return "have total length of " + _gr_length_phrase_en(b)


def _gr_area_phrase_ru(b: BoundSpec) -> str:
    return _gr_bound_phrase_ru(b, _gr_fmt_int, "км²")


def _gr_area_phrase_en(b: BoundSpec) -> str:
    return _gr_bound_phrase_en(b, _gr_fmt_int, "square kilometres")


def _gr_area_clause_ru(b: BoundSpec) -> str:
    return "площадью " + _gr_area_phrase_ru(b)


def _gr_area_clause_en(b: BoundSpec) -> str:
    return "with area of " + _gr_area_phrase_en(b)


def _gr_elevation_phrase_ru(b: BoundSpec) -> str:
    return _gr_bound_phrase_ru(b, _gr_fmt_int, "м")


def _gr_elevation_phrase_en(b: BoundSpec) -> str:
    return _gr_bound_phrase_en(b, _gr_fmt_int, "m")


def _gr_numeric_modes_for(complexity: str) -> Tuple[str, ...]:
    # Simple tasks stay readable but still include max/range; harder tasks use
    # the full mix more aggressively.
    if complexity in {"L1", "L2"}:
        return ("min", "max", "range")
    return ("min", "max", "range", "range")




# Lightweight pre-WDQS diversity helpers.
# Region-like anchors are rotated per (complexity, template_id), so repeated
# template calls do not keep probing the same federal subject with a different
# numeric comparator. This is intentionally not a hard post-generation skip.
def _gr_pick_region_for_template(
    rng: random.Random,
    complexity: str,
    template_id: str,
    regions: Sequence[Dict[str, Any]],
) -> Dict[str, Any]:
    if not regions:
        raise ValueError("empty region pool")
    ctx = globals().get("_GEO_RU_FAST_DIVERSITY_CONTEXT")
    if isinstance(ctx, dict):
        key = (str(complexity), str(template_id))
        region_orders = ctx.setdefault("region_orders", {})
        region_cursors = ctx.setdefault("region_cursors", defaultdict(int))
        if key not in region_orders:
            order = list(regions)
            rng.shuffle(order)
            region_orders[key] = order
        order = region_orders[key]
        pos = int(region_cursors[key])
        region_cursors[key] = pos + 1
        return order[pos % len(order)]
    return rng.choice(list(regions))


def _gr_tpl_cities_in_region_population(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L1", "L2"}:
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_city_region_population", REGION_SPECS)
    pop_bound = _gr_pick_bound(rng, region["city_pop_min"], modes=_gr_numeric_modes_for(complexity))
    requested = 5 if complexity == "L1" else 4
    q_ru = f"Назови {requested} {_gr_ru_count(requested, 'город', 'города', 'городов')} России, расположенных в {region['ru_prep']}, {_gr_population_clause_ru(pop_bound)}."
    q_en = f"Name {requested} cities in Russia located in {region['label_en']} {_gr_population_clause_en(pop_bound)}."
    where = [
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", pop_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_city_region_population",
        template_family="cities_admin",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "country": COUNTRY_RUSSIA,
            "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
            **_gr_bound_constraints(pop_bound, "population_min", "population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_noncapital_cities_in_region(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_noncapital_city_region_population", REGION_SPECS)
    pop_bound = _gr_pick_bound(rng, region["city_pop_min"], modes=("min", "max", "range"))
    requested = 4
    q_ru = f"Назови {requested} города России в {region['ru_prep']} {_gr_population_clause_ru(pop_bound)}, которые не являются административным центром этого субъекта."
    q_en = f"Name {requested} Russian cities in {region['label_en']} {_gr_population_clause_en(pop_bound)} that are not the administrative centre of that federal subject."
    where = [
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        f"wd:{region['qid']} wdt:P36 ?regional_capital .",
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", pop_bound),
        "FILTER(?item != ?regional_capital) .",
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_noncapital_city_region_population",
        template_family="cities_admin",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "country": COUNTRY_RUSSIA,
            "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
            "exclude_administrative_centre_of_entity": _gr_entity(region["qid"], region["label_en"]),
            **_gr_bound_constraints(pop_bound, "population_min", "population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_rivers_in_russia_length(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L1", "L2"}:
        return None
    values = [300, 500, 800, 1000, 1500] if complexity == "L1" else [500, 700, 1000, 1500, 2000]
    # Max-only river queries can be broad; the 101-probe gate will reject any
    # that still produce more than 100 fully-labelled golds.
    length_bound = _gr_pick_bound(rng, values, modes=("min", "max", "range"))
    requested = 5 if complexity == "L1" else 4
    q_ru = f"Назови {requested} {_gr_ru_count(requested, 'реку', 'реки', 'рек')}, которые протекают по территории России, {_gr_length_clause_ru(length_bound)}."
    q_en = f"Name {requested} rivers that flow through Russia and {_gr_length_clause_en(length_bound)}."
    where = [
        _gr_type_line("item", Q_RIVER),
        _gr_country_russia_line("item"),
        *_gr_quantity_lines("item", "P2043", "length_km", "length_km"),
        *_gr_bound_filters("length_km", length_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_river_country_length",
        template_family="rivers",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "river",
            "country": COUNTRY_RUSSIA,
            **_gr_bound_constraints(length_bound, "length_min_km", "length_max_km"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_lakes_in_region_area(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L1", "L2"}:
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_lake_region_area", REGION_SPECS)
    area_bound = _gr_pick_bound(rng, region["lake_area_min"], modes=("min", "max", "range"))
    requested = 5 if complexity == "L1" else 4
    q_ru = f"Назови {requested} {_gr_ru_count(requested, 'озеро', 'озера', 'озёр')} России, расположенных в {region['ru_prep']}, {_gr_area_clause_ru(area_bound)}."
    q_en = f"Name {requested} lakes in Russia located in {region['label_en']} {_gr_area_clause_en(area_bound)}."
    where = [
        *_gr_natural_lake_lines("item"),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        *_gr_quantity_lines("item", "P2046", "area_km2", "area_km2"),
        *_gr_bound_filters("area_km2", area_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_lake_region_area",
        template_family="lakes",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "lake",
            "country": COUNTRY_RUSSIA,
            "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
            **_gr_bound_constraints(area_bound, "area_min_km2", "area_max_km2"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_admin_centers_population(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L2":
        return None
    pop_bound = _gr_pick_bound(rng, [100_000, 250_000, 500_000, 800_000, 1_000_000], modes=("min", "max", "range"))
    requested = 4
    q_ru = f"Назови {requested} административных центра субъектов РФ, где население города {_gr_population_phrase_ru(pop_bound)}."
    q_en = f"Name {requested} administrative centres of Russian federal subjects whose city population is {_gr_population_phrase_en(pop_bound)}."
    where = [
        *_gr_federal_subject_lines("subject"),
        _gr_country_russia_line("subject"),
        "?subject wdt:P36 ?item .",
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", pop_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_admin_center_subject_population",
        template_family="admin_centers",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "city_role": "administrative centre of a Russian federal subject",
            "federal_subject_country": COUNTRY_RUSSIA,
            **_gr_bound_constraints(pop_bound, "population_min", "population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


## 7. L3–L5 multi-hop templates

Hard templates connect the answer entity through another Wikidata entity: federal subject → capital, subject → area, city → river → mouth, protected area → subject → mountain, and so on.


In [7]:
def _gr_tpl_admin_centers_subject_area(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4"}:
        return None
    area_values = [50_000, 100_000, 200_000, 500_000, 1_000_000] if complexity == "L3" else [100_000, 300_000, 700_000, 1_000_000, 3_000_000]
    area_bound = _gr_pick_bound(rng, area_values, modes=_gr_numeric_modes_for(complexity))
    pop_bound = _gr_pick_bound(rng, [100_000, 250_000, 500_000, 800_000, 1_000_000], modes=("min", "max", "range"))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} административных центра субъектов РФ: площадь соответствующего субъекта должна быть {_gr_area_phrase_ru(area_bound)}, а население города — {_gr_population_phrase_ru(pop_bound)}."
    q_en = f"Name {requested} administrative centres of Russian federal subjects where the federal subject area is {_gr_area_phrase_en(area_bound)} and the city population is {_gr_population_phrase_en(pop_bound)}."
    where = [
        *_gr_federal_subject_lines("subject"),
        _gr_country_russia_line("subject"),
        "?subject wdt:P36 ?item .",
        *_gr_quantity_lines("subject", "P2046", "subject_area_km2", "area_km2"),
        *_gr_bound_filters("subject_area_km2", area_bound),
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", pop_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_admin_center_subject_area_city_population",
        template_family="admin_centers_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "city_role": "administrative centre of a Russian federal subject",
            "federal_subject_country": COUNTRY_RUSSIA,
            **_gr_bound_constraints(area_bound, "federal_subject_area_min_km2", "federal_subject_area_max_km2"),
            **_gr_bound_constraints(pop_bound, "city_population_min", "city_population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_cities_subject_capital_population(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4"}:
        return None
    cap_values = [300_000, 500_000, 800_000, 1_000_000, 1_200_000]
    capital_pop_bound = _gr_pick_bound(rng, cap_values, modes=_gr_numeric_modes_for(complexity))
    city_pop_bound = _gr_pick_bound(rng, [20_000, 50_000, 100_000, 200_000, 300_000], modes=("min", "max", "range"))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} города России {_gr_population_clause_ru(city_pop_bound)}, расположенные в субъектах РФ, где население административного центра {_gr_population_phrase_ru(capital_pop_bound)}. Сам административный центр не засчитывай."
    q_en = f"Name {requested} Russian cities {_gr_population_clause_en(city_pop_bound)} that are located in federal subjects whose administrative centre population is {_gr_population_phrase_en(capital_pop_bound)}. Do not count the administrative centre itself."
    where = [
        *_gr_federal_subject_lines("subject"),
        _gr_country_russia_line("subject"),
        "?subject wdt:P36 ?capital .",
        "?capital wdt:P1082 ?capital_population .",
        *_gr_bound_filters("capital_population", capital_pop_bound),
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        "?item wdt:P131/wdt:P131* ?subject .",
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", city_pop_bound),
        "FILTER(?item != ?capital) .",
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_city_in_subject_with_large_capital",
        template_family="cities_subject_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "country": COUNTRY_RUSSIA,
            **_gr_bound_constraints(capital_pop_bound, "federal_subject_capital_population_min", "federal_subject_capital_population_max"),
            "exclude_subject_capital": True,
            **_gr_bound_constraints(city_pop_bound, "city_population_min", "city_population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_protected_areas_subject_capital(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4"}:
        return None
    capital_pop_bound = _gr_pick_bound(rng, [300_000, 500_000, 800_000, 1_000_000], modes=("min", "max", "range"))
    year_max = rng.choice([1980, 1990, 2000, 2010]) if complexity == "L3" else rng.choice([1970, 1985, 2000])
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} охраняемые территории России, созданные не позже {year_max} года и расположенные в субъектах РФ, где население административного центра {_gr_population_phrase_ru(capital_pop_bound)}."
    q_en = f"Name {requested} protected areas in Russia established no later than {year_max} and located in federal subjects whose administrative centre population is {_gr_population_phrase_en(capital_pop_bound)}."
    where = [
        _gr_type_line("item", Q_PROTECTED_AREA),
        _gr_country_russia_line("item"),
        "?item wdt:P131/wdt:P131* ?subject .",
        *_gr_federal_subject_lines("subject"),
        _gr_country_russia_line("subject"),
        "?subject wdt:P36 ?capital .",
        "?capital wdt:P1082 ?capital_population .",
        *_gr_bound_filters("capital_population", capital_pop_bound),
        "?item wdt:P571 ?inception .",
        _gr_year_max_line("inception", year_max),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_protected_area_subject_capital_inception",
        template_family="protected_areas_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "protected area",
            "country": COUNTRY_RUSSIA,
            **_gr_bound_constraints(capital_pop_bound, "federal_subject_capital_population_min", "federal_subject_capital_population_max"),
            "inception_year_max": year_max,
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_rivers_mouth_to_sea_length(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    sea_group = rng.choice(SEA_GROUPS)
    values = [100, 300, 500, 700, 1000] if complexity == "L4" else [300, 500, 800, 1000, 1500]
    length_bound = _gr_pick_bound(rng, values, modes=_gr_numeric_modes_for(complexity))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} {_gr_ru_count(requested, 'реку', 'реки', 'рек')}, которые протекают по территории России, {_gr_length_clause_ru(length_bound)} и впадают в {sea_group['ru']}."
    q_en = f"Name {requested} rivers that flow through Russia, {_gr_length_clause_en(length_bound)}, and flow into {sea_group['label_en']}."
    where = [
        _gr_type_line("item", Q_RIVER),
        _gr_country_russia_line("item"),
        _gr_values_qids("mouth_water", [x["qid"] for x in sea_group["entities"]]),
        "?item wdt:P403 ?mouth_water .",
        *_gr_quantity_lines("item", "P2043", "length_km", "length_km"),
        *_gr_bound_filters("length_km", length_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_river_mouth_sea_length",
        template_family="rivers_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "river",
            "country": COUNTRY_RUSSIA,
            "mouth_of_watercourse_one_of": [_gr_entity(x["qid"], x["label_en"]) for x in sea_group["entities"]],
            **_gr_bound_constraints(length_bound, "length_min_km", "length_max_km"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_cities_on_rivers_in_large_capital_subjects(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    capital_pop_bound = _gr_pick_bound(rng, [500_000, 800_000, 1_000_000, 1_200_000], modes=("min", "max", "range"))
    city_values = [50_000, 100_000, 200_000, 300_000] if complexity == "L4" else [100_000, 200_000, 300_000, 500_000]
    city_pop_bound = _gr_pick_bound(rng, city_values, modes=_gr_numeric_modes_for(complexity))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} города России {_gr_population_clause_ru(city_pop_bound)}, которые стоят на реке и расположены в субъектах РФ, где население административного центра {_gr_population_phrase_ru(capital_pop_bound)}."
    q_en = f"Name {requested} Russian cities {_gr_population_clause_en(city_pop_bound)} that are located on a river and lie in federal subjects whose administrative centre population is {_gr_population_phrase_en(capital_pop_bound)}."
    where = [
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        "?item wdt:P131/wdt:P131* ?subject .",
        *_gr_federal_subject_lines("subject"),
        _gr_country_russia_line("subject"),
        "?subject wdt:P36 ?capital .",
        "?capital wdt:P1082 ?capital_population .",
        *_gr_bound_filters("capital_population", capital_pop_bound),
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", city_pop_bound),
        "?item wdt:P206 ?river .",
        _gr_type_line("river", Q_RIVER),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_city_on_river_in_subject_with_large_capital",
        template_family="cities_rivers_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "country": COUNTRY_RUSSIA,
            "located_on_physical_feature_type": "river",
            **_gr_bound_constraints(capital_pop_bound, "federal_subject_capital_population_min", "federal_subject_capital_population_max"),
            **_gr_bound_constraints(city_pop_bound, "city_population_min", "city_population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_cities_on_tributaries(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    river = _gr_pick_region_for_template(rng, complexity, "geo_ru_cities_on_tributaries", RIVER_SEEDS)
    pop_bound = _gr_pick_bound(rng, [50_000, 100_000, 200_000, 300_000, 500_000], modes=("min", "max", "range"))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} города России {_gr_population_clause_ru(pop_bound)}, которые стоят на реках, впадающих в {river['ru_dat']}."
    q_en = f"Name {requested} Russian cities {_gr_population_clause_en(pop_bound)} that are located on rivers flowing into the {river['label_en']}."
    where = [
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", pop_bound),
        "?item wdt:P206 ?tributary .",
        _gr_type_line("tributary", Q_RIVER),
        f"?tributary wdt:P403 wd:{river['qid']} .",
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_city_on_tributary_of_major_river",
        template_family="cities_rivers_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "country": COUNTRY_RUSSIA,
            "located_on_river_that_flows_into": _gr_entity(river["qid"], river["label_en"]),
            **_gr_bound_constraints(pop_bound, "city_population_min", "city_population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_protected_areas_same_subject_as_high_mountain(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity != "L5":
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_protected_area_same_subject_high_mountain", ELEVATED_REGION_SPECS)
    elev_bound = _gr_pick_bound(rng, [1500, 2000, 2500, 3000, 3500, 4000], modes=("min", "max", "range"))
    year_max = rng.choice([1990, 2000, 2010])
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} охраняемые территории России в {region['ru_prep']}, созданные не позже {year_max} года, если в этом же субъекте есть гора или вулкан высотой {_gr_elevation_phrase_ru(elev_bound)}."
    q_en = f"Name {requested} protected areas in Russia located in {region['label_en']} and established no later than {year_max}, where the same federal subject has a mountain or volcano with elevation {_gr_elevation_phrase_en(elev_bound)}."
    where = [
        _gr_type_line("item", Q_PROTECTED_AREA),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        "?item wdt:P571 ?inception .",
        _gr_year_max_line("inception", year_max),
        _gr_values_qids("high_geo_type", [Q_MOUNTAIN, Q_VOLCANO]),
        "?high_geo wdt:P31/wdt:P279* ?high_geo_type .",
        _gr_country_russia_line("high_geo"),
        f"?high_geo wdt:P131/wdt:P131* wd:{region['qid']} .",
        *_gr_quantity_lines("high_geo", "P2044", "elevation_m", "elevation_m"),
        *_gr_bound_filters("elevation_m", elev_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_protected_area_same_subject_as_high_mountain",
        template_family="protected_areas_mountains_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "protected area",
            "country": COUNTRY_RUSSIA,
            "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
            "same_administrative_entity_has_feature_type_one_of": ["mountain", "volcano"],
            **_gr_bound_constraints(elev_bound, "same_administrative_entity_feature_elevation_min_m", "same_administrative_entity_feature_elevation_max_m"),
            "inception_year_max": year_max,
        },
        where_lines=where,
        requested_count=requested,
    )


def _gr_tpl_cities_subject_with_large_lake(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_city_same_subject_as_large_lake", REGION_SPECS)
    lake_area_bound = _gr_pick_bound(rng, region["lake_area_min"], modes=("min", "max", "range"))
    city_values = [20_000, 50_000, 100_000, 200_000] if complexity == "L4" else [50_000, 100_000, 200_000, 300_000]
    city_pop_bound = _gr_pick_bound(rng, city_values, modes=_gr_numeric_modes_for(complexity))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} города России в {region['ru_prep']} {_gr_population_clause_ru(city_pop_bound)}, если в этом же субъекте есть озеро {_gr_area_clause_ru(lake_area_bound)}."
    q_en = f"Name {requested} Russian cities in {region['label_en']} {_gr_population_clause_en(city_pop_bound)}, where the same federal subject contains a lake {_gr_area_clause_en(lake_area_bound)}."
    where = [
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", city_pop_bound),
        *_gr_natural_lake_lines("lake"),
        _gr_country_russia_line("lake"),
        f"?lake wdt:P131/wdt:P131* wd:{region['qid']} .",
        *_gr_quantity_lines("lake", "P2046", "lake_area_km2", "area_km2"),
        *_gr_bound_filters("lake_area_km2", lake_area_bound),
    ]
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_city_same_subject_as_large_lake",
        template_family="cities_lakes_multihop",
        q_ru=q_ru, q_en=q_en,
        constraints={
            "answer_type": "city",
            "country": COUNTRY_RUSSIA,
            "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
            **_gr_bound_constraints(lake_area_bound, "same_administrative_entity_has_lake_area_min_km2", "same_administrative_entity_has_lake_area_max_km2"),
            **_gr_bound_constraints(city_pop_bound, "city_population_min", "city_population_max"),
        },
        where_lines=where,
        requested_count=requested,
    )

# --- v18 diversity patch: nature-heavy advanced templates -------------------
# These definitions intentionally override/extend a few v17 functions before the
# template registry is built in the next cell.

def _gr_geo_type_values_line(var: str, qids: Sequence[str]) -> str:
    vals = " ".join(f"wd:{q}" for q in dict.fromkeys(qids) if _QID_RE_GEO_RU.fullmatch(str(q)))
    return f"VALUES ?{var}_type {{ {vals} }}"


def _gr_mountain_or_volcano_lines(var: str) -> List[str]:
    return [
        _gr_geo_type_values_line(var, [Q_MOUNTAIN, Q_VOLCANO]),
        f"?{var} wdt:P31/wdt:P279* ?{var}_type .",
    ]


def _gr_tpl_mountains_or_volcanoes_region_elevation(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_mountain_or_volcano_region_elevation", ELEVATED_REGION_SPECS)
    elev_values = [500, 1000, 1500, 2000, 3000, 4000] if complexity == "L3" else [1000, 1500, 2000, 3000, 4000, 5000]
    elev_bound = _gr_pick_bound(rng, elev_values, modes=_gr_numeric_modes_for(complexity))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} горы или вулкана России, расположенные в {region['ru_prep']}, высотой {_gr_elevation_phrase_ru(elev_bound)}."
    q_en = f"Name {requested} mountains or volcanoes in Russia located in {region['label_en']} with elevation {_gr_elevation_phrase_en(elev_bound)}."
    where = [
        *_gr_mountain_or_volcano_lines("item"),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        *_gr_quantity_lines("item", "P2044", "elevation_m", "elevation_m"),
        *_gr_bound_filters("elevation_m", elev_bound),
    ]
    constraints = {
        "answer_type": "mountain or volcano",
        "country": COUNTRY_RUSSIA,
        "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
    }
    if elev_bound.get("min") is not None:
        constraints["elevation_min_m"] = int(elev_bound["min"])
    if elev_bound.get("max") is not None:
        constraints["elevation_max_m"] = int(elev_bound["max"])
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_mountain_or_volcano_region_elevation",
        template_family="mountains_volcanoes",
        q_ru=q_ru, q_en=q_en,
        constraints=constraints,
        requested_count=requested,
        where_lines=where,
    )


def _gr_inception_year_bound_for(complexity: str, rng: random.Random) -> BoundSpec:
    values = [1950, 1970, 1980, 1990, 2000, 2010]
    # “max” means established no later than year; “min” means no earlier than.
    return _gr_pick_bound(rng, values, modes=_gr_numeric_modes_for(complexity))


def _gr_year_bound_filters(date_var: str, b: BoundSpec) -> List[str]:
    out: List[str] = []
    if b.get("min") is not None:
        out.append(_gr_year_min_line(date_var, int(b["min"])))
    if b.get("max") is not None:
        out.append(_gr_year_max_line(date_var, int(b["max"])))
    return out


def _gr_year_phrase_ru(b: BoundSpec) -> str:
    if b.get("min") is not None and b.get("max") is not None:
        return f"созданные с {int(b['min'])} по {int(b['max'])} год"
    if b.get("max") is not None:
        return f"созданные не позже {int(b['max'])} года"
    if b.get("min") is not None:
        return f"созданные не раньше {int(b['min'])} года"
    return "с известным годом создания"


def _gr_year_phrase_en(b: BoundSpec) -> str:
    if b.get("min") is not None and b.get("max") is not None:
        return f"established between {int(b['min'])} and {int(b['max'])}"
    if b.get("max") is not None:
        return f"established no later than {int(b['max'])}"
    if b.get("min") is not None:
        return f"established no earlier than {int(b['min'])}"
    return "with known inception year"


def _gr_tpl_protected_areas_region_inception(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_protected_areas_region_inception", REGION_SPECS)
    year_bound = _gr_inception_year_bound_for(complexity, rng)
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} охраняемые территории России, расположенные в {region['ru_prep']}, {_gr_year_phrase_ru(year_bound)}."
    q_en = f"Name {requested} protected areas in Russia located in {region['label_en']} and {_gr_year_phrase_en(year_bound)}."
    where = [
        _gr_type_line("item", Q_PROTECTED_AREA),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        "?item wdt:P571 ?inception .",
        *_gr_year_bound_filters("inception", year_bound),
    ]
    constraints = {
        "answer_type": "protected area",
        "country": COUNTRY_RUSSIA,
        "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
    }
    if year_bound.get("min") is not None:
        constraints["inception_year_min"] = int(year_bound["min"])
    if year_bound.get("max") is not None:
        constraints["inception_year_max"] = int(year_bound["max"])
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_protected_areas_region_inception",
        template_family="protected_areas",
        q_ru=q_ru, q_en=q_en,
        constraints=constraints,
        requested_count=requested,
        where_lines=where,
    )


def _gr_tpl_rivers_mouth_to_sea_length(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L3", "L4", "L5"}:
        return None
    sea_group = rng.choice(SEA_GROUPS)
    values = [100, 300, 500, 700, 1000] if complexity in {"L3", "L4"} else [300, 500, 800, 1000, 1500]
    length_bound = _gr_pick_bound(rng, values, modes=_gr_numeric_modes_for(complexity))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} {_gr_ru_count(requested, 'реку', 'реки', 'рек')}, которые протекают по территории России, {_gr_length_clause_ru(length_bound)} и впадают в {sea_group['ru']}."
    q_en = f"Name {requested} rivers that flow through Russia, {_gr_length_clause_en(length_bound)}, and flow into {sea_group['label_en']}."
    where = [
        _gr_type_line("item", Q_RIVER),
        _gr_country_russia_line("item"),
        _gr_values_qids("mouth_water", [x["qid"] for x in sea_group["entities"]]),
        "?item wdt:P403 ?mouth_water .",
        *_gr_quantity_lines("item", "P2043", "length_km", "length_km"),
        *_gr_bound_filters("length_km", length_bound),
    ]
    constraints = {
        "answer_type": "river",
        "country": COUNTRY_RUSSIA,
        "mouth_of_the_watercourse": [x["label_en"] for x in sea_group["entities"]],
    }
    if length_bound.get("min") is not None:
        constraints["length_min_km"] = int(length_bound["min"])
    if length_bound.get("max") is not None:
        constraints["length_max_km"] = int(length_bound["max"])
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_rivers_mouth_to_sea_length",
        template_family="rivers_hydrography",
        q_ru=q_ru, q_en=q_en,
        constraints=constraints,
        requested_count=requested,
        where_lines=where,
    )


def _gr_tpl_cities_on_tributaries(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    river = _gr_pick_region_for_template(rng, complexity, "geo_ru_cities_on_tributaries", RIVER_SEEDS)
    pop_values = [50_000, 100_000, 200_000, 300_000] if complexity == "L4" else [50_000, 100_000, 200_000, 300_000, 500_000]
    pop_bound = _gr_pick_bound(rng, pop_values, modes=("min", "max", "range"))
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} города России {_gr_population_clause_ru(pop_bound)}, которые стоят на реках, впадающих в {river['ru_dat']}."
    q_en = f"Name {requested} Russian cities {_gr_population_clause_en(pop_bound)} that are located on rivers flowing into the {river['label_en']}."
    where = [
        *_gr_strict_city_lines("item"),
        _gr_country_russia_line("item"),
        "?item wdt:P1082 ?population .",
        *_gr_bound_filters("population", pop_bound),
        "?item wdt:P206 ?tributary .",
        _gr_type_line("tributary", Q_RIVER),
        f"?tributary wdt:P403 wd:{river['qid']} .",
    ]
    constraints = {
        "answer_type": "city",
        "country": COUNTRY_RUSSIA,
        "located_on_river_that_flows_into": _gr_entity(river["qid"], river["label_en"]),
    }
    if pop_bound.get("min") is not None:
        constraints["population_min"] = int(pop_bound["min"])
    if pop_bound.get("max") is not None:
        constraints["population_max"] = int(pop_bound["max"])
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_city_on_tributary_of_major_river",
        template_family="cities_hydrography",
        q_ru=q_ru, q_en=q_en,
        constraints=constraints,
        requested_count=requested,
        where_lines=where,
    )


def _gr_tpl_protected_areas_same_subject_as_high_mountain(complexity: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    if complexity not in {"L4", "L5"}:
        return None
    region = _gr_pick_region_for_template(rng, complexity, "geo_ru_protected_area_same_subject_high_mountain", ELEVATED_REGION_SPECS)
    elev_bound = _gr_pick_bound(rng, [1500, 2000, 2500, 3000, 3500, 4000], modes=("min", "max", "range"))
    year_max = rng.choice([1990, 2000, 2010])
    requested = GEO_RU_REQUESTED_BY_LEVEL[complexity]
    q_ru = f"Назови {requested} охраняемые территории России в {region['ru_prep']}, созданные не позже {year_max} года, если в этом же субъекте есть гора или вулкан высотой {_gr_elevation_phrase_ru(elev_bound)}."
    q_en = f"Name {requested} protected areas in Russia located in {region['label_en']} and established no later than {year_max}, where the same federal subject has a mountain or volcano with elevation {_gr_elevation_phrase_en(elev_bound)}."
    where = [
        _gr_type_line("item", Q_PROTECTED_AREA),
        _gr_country_russia_line("item"),
        f"?item wdt:P131/wdt:P131* wd:{region['qid']} .",
        "?item wdt:P571 ?inception .",
        _gr_year_max_line("inception", year_max),
        f"?mountain wdt:P131/wdt:P131* wd:{region['qid']} .",
        *_gr_mountain_or_volcano_lines("mountain"),
        *_gr_quantity_lines("mountain", "P2044", "elevation_m", "elevation_m"),
        *_gr_bound_filters("elevation_m", elev_bound),
    ]
    constraints = {
        "answer_type": "protected area",
        "country": COUNTRY_RUSSIA,
        "located_in_administrative_entity": _gr_entity(region["qid"], region["label_en"]),
        "inception_year_max": int(year_max),
    }
    if elev_bound.get("min") is not None:
        constraints["same_subject_mountain_or_volcano_elevation_min_m"] = int(elev_bound["min"])
    if elev_bound.get("max") is not None:
        constraints["same_subject_mountain_or_volcano_elevation_max_m"] = int(elev_bound["max"])
    return _gr_finalize_wdqs_example(
        idx=idx, complexity=complexity,
        template_id="geo_ru_protected_area_same_subject_high_mountain",
        template_family="protected_areas_mountains",
        q_ru=q_ru, q_en=q_en,
        constraints=constraints,
        requested_count=requested,
        where_lines=where,
    )



## 8. Template registry and one-example generator

The generator shuffles templates and parameters, skips empty/unstable WDQS results, and returns a failure stub only after all attempts are exhausted.


In [8]:
GEO_RU_TEMPLATE_FNS: Dict[str, List[Callable[[str, int, random.Random], Optional[BenchmarkExample]]]] = {
    "L1": [
        _gr_tpl_lakes_in_region_area,
        _gr_tpl_cities_in_region_population,
        _gr_tpl_rivers_in_russia_length,
    ],
    "L2": [
        _gr_tpl_noncapital_cities_in_region,
        _gr_tpl_lakes_in_region_area,
        _gr_tpl_rivers_in_russia_length,
        _gr_tpl_cities_in_region_population,
    ],
    "L3": [
        _gr_tpl_mountains_or_volcanoes_region_elevation,
        _gr_tpl_protected_areas_region_inception,
        _gr_tpl_rivers_mouth_to_sea_length,
        _gr_tpl_protected_areas_subject_capital,
    ],
    "L4": [
        _gr_tpl_rivers_mouth_to_sea_length,
        _gr_tpl_protected_areas_same_subject_as_high_mountain,
        _gr_tpl_mountains_or_volcanoes_region_elevation,
        _gr_tpl_cities_subject_with_large_lake,
        _gr_tpl_cities_on_tributaries,
        _gr_tpl_protected_areas_region_inception,
        _gr_tpl_cities_on_rivers_in_large_capital_subjects,
    ],
    "L5": [
        _gr_tpl_rivers_mouth_to_sea_length,
        _gr_tpl_cities_on_tributaries,
        _gr_tpl_protected_areas_same_subject_as_high_mountain,
        _gr_tpl_cities_subject_with_large_lake,
        _gr_tpl_mountains_or_volcanoes_region_elevation,
        _gr_tpl_protected_areas_region_inception,
        _gr_tpl_cities_on_rivers_in_large_capital_subjects,
    ],
}


GEO_RU_TEMPLATE_ID_TO_FN: Dict[str, Callable[[str, int, random.Random], Optional[BenchmarkExample]]] = {
    "geo_ru_city_region_population": _gr_tpl_cities_in_region_population,
    "geo_ru_noncapital_city_region_population": _gr_tpl_noncapital_cities_in_region,
    "geo_ru_river_country_length": _gr_tpl_rivers_in_russia_length,
    "geo_ru_lake_region_area": _gr_tpl_lakes_in_region_area,
    "geo_ru_admin_center_subject_population": _gr_tpl_admin_centers_population,
    "geo_ru_admin_centers_subject_area": _gr_tpl_admin_centers_subject_area,
    "geo_ru_mountain_or_volcano_region_elevation": _gr_tpl_mountains_or_volcanoes_region_elevation,
    "geo_ru_protected_areas_region_inception": _gr_tpl_protected_areas_region_inception,
    "geo_ru_cities_subject_capital_population": _gr_tpl_cities_subject_capital_population,
    "geo_ru_protected_areas_subject_capital": _gr_tpl_protected_areas_subject_capital,
    "geo_ru_rivers_mouth_to_sea_length": _gr_tpl_rivers_mouth_to_sea_length,
    "geo_ru_cities_on_rivers_in_large_capital_subjects": _gr_tpl_cities_on_rivers_in_large_capital_subjects,
    "geo_ru_cities_on_tributaries": _gr_tpl_cities_on_tributaries,
    "geo_ru_protected_area_same_subject_high_mountain": _gr_tpl_protected_areas_same_subject_as_high_mountain,
    "geo_ru_city_same_subject_as_large_lake": _gr_tpl_cities_subject_with_large_lake,
}

def _gr_desired_numeric_mode(complexity: str, accepted_index_within_level: int) -> Optional[str]:
    plan = GEO_RU_NUMERIC_MODE_PLAN.get(complexity)
    if not plan:
        return None
    return plan[int(accepted_index_within_level) % len(plan)]


def _gr_desired_template_id(complexity: str, accepted_index_within_level: int) -> Optional[str]:
    plan = GEO_RU_TEMPLATE_PLAN_BY_LEVEL.get(complexity)
    if not plan:
        return None
    return plan[int(accepted_index_within_level) % len(plan)]


def generate_geo_ru_example(
    complexity: str,
    idx: int,
    rng: Optional[random.Random] = None,
    max_attempts: Optional[int] = None,
    forced_numeric_mode: Optional[str] = None,
    forced_template_id: Optional[str] = None,
) -> BenchmarkExample:
    """Generate one example with bounded WDQS cost.

    Final speed rule: if a template is requested by the dataset plan, only that
    template is probed.  Older versions silently tried several other templates
    inside one progress-bar step, which made generation look frozen and caused
    near-duplicate fallbacks.  Fallbacks now live one level higher in the dataset
    loop, where they are visible and limited.
    """
    rng = rng or GEO_RU_RNG
    max_attempts = int(max_attempts or globals().get("GEO_RU_GENERATOR_MAX_ATTEMPTS", 3))
    all_fns = list(GEO_RU_TEMPLATE_FNS.get(complexity, []))
    if not all_fns:
        raise ValueError(f"Unknown geo_ru complexity: {complexity}")

    if forced_template_id:
        fn = GEO_RU_TEMPLATE_ID_TO_FN.get(forced_template_id)
        if fn is None:
            raise ValueError(f"Unknown geo_ru template_id: {forced_template_id}")
        fns = [fn]
    else:
        fns = [rng.choice(all_fns)]

    mode_order: List[Optional[str]]
    if forced_numeric_mode in {"min", "max", "range"}:
        mode_order = [forced_numeric_mode]
    else:
        mode_order = [None]

    old_mode = globals().get("_GEO_RU_FORCED_NUMERIC_MODE")
    try:
        for mode in mode_order:
            globals()["_GEO_RU_FORCED_NUMERIC_MODE"] = mode
            for attempt_i in range(1, max_attempts + 1):
                fn = fns[0]
                try:
                    ex = fn(complexity, idx, rng)
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if globals().get("DEBUG_GENERATOR_ERRORS", False):
                        print(f"[geo_ru] {getattr(fn, '__name__', fn)} failed: {type(e).__name__}: {e}")
                    ex = None

                if ex is not None and ex.gold_answer_qids and ex.query_text_en:
                    meta = getattr(ex, "gold_collection_meta", None) or {}
                    meta["generator_attempts_for_record"] = attempt_i
                    meta["planned_template_id"] = forced_template_id
                    meta["planned_numeric_mode"] = forced_numeric_mode
                    meta["actual_numeric_mode"] = _gr_constraint_numeric_mode(ex.constraints or {})
                    ex.gold_collection_meta = meta
                    return ex
    finally:
        globals()["_GEO_RU_FORCED_NUMERIC_MODE"] = old_mode

    return _gr_failed_example(complexity, idx)


# Backward-compatible aliases used by some older driver notebooks.
generate_geo_example = generate_geo_ru_example
if "DOMAIN_GENERATORS" in globals():
    DOMAIN_GENERATORS["geo_ru"] = generate_geo_ru_example

print("geo_ru generator ready:", {k: len(v) for k, v in GEO_RU_TEMPLATE_FNS.items()})




geo_ru generator ready: {'L1': 3, 'L2': 4, 'L3': 4, 'L4': 7, 'L5': 7}


## 9. Non-network sanity checks

These checks fail fast if an old/incorrect version of the notebook is being executed. They do not call WDQS.


In [9]:
def _gr_run_sanity_checks() -> None:
    test_query = _gr_build_select_query([
        _gr_type_line("item", Q_RIVER),
        _gr_country_russia_line("item"),
    ], item_var="item", limit=GEO_RU_PROBE_LIMIT)

    assert "OPTIONAL {" not in test_query, "Old optional-label logic is still present"
    old_bound_marker = "BOUND(?itemLabelRu)" + " || " + "BOUND(?itemLabelEn)"
    assert old_bound_marker not in test_query, "Old ru/en fallback filter is still present"
    assert '?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") .' in test_query
    assert '?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") .' in test_query
    assert "LIMIT 101" in test_query, "Gold probe must use LIMIT 101"


    # v17: federal-subject templates must use all Russian federal subject types,
    # not only Q835714 ("oblast of Russia").  Otherwise republics/krais/federal
    # cities are silently missing from admin-centre and multihop golds.
    fs_lines = "\n".join(_gr_federal_subject_lines("subject"))
    for qid in (Q_OBLAST_RUSSIA, Q_REPUBLIC_RUSSIA, Q_FEDERAL_CITY_RUSSIA, Q_KRAI_RUSSIA, Q_AUTONOMOUS_OBLAST_RUSSIA, Q_AUTONOMOUS_OKRUG_RUSSIA):
        assert f"wd:{qid}" in fs_lines
    assert "wdt:P31/wdt:P279* wd:Q835714" not in "\n".join(_gr_federal_subject_lines("subject"))

    lake_where = "\n".join(_gr_natural_lake_lines("item"))
    assert f"wd:{Q_RESERVOIR}" in lake_where, "Reservoir exclusion is missing"
    assert f"wd:{Q_WATER_RESERVOIR}" in lake_where, "Water-reservoir exclusion is missing"
    assert "водохранилище" in lake_where.lower(), "Russian reservoir label exclusion is missing"
    assert "reservoir" in lake_where.lower(), "English reservoir label exclusion is missing"


    city_where = "\n".join(_gr_strict_city_lines("item"))
    assert f"wd:{Q_VILLAGE}" in city_where, "Village subclass exclusion is missing"
    assert "village|settlement|hamlet" in city_where, "City leakage label exclusion is missing"
    assert "деревня|село" in city_where, "Russian settlement label exclusion is missing"
    assert _gr_duplicate_gold_labels([
        {"qid": "Q1", "label_ru": "Горное озеро", "label_en": "Gornoe"},
        {"qid": "Q2", "label_ru": "Горное озеро", "label_en": "Gornoe"},
    ]), "Duplicate public gold labels must be detected"

    assert GEO_RU_MAX_ACCEPTED_GOLD == 100, "Accepted gold max must be 100"
    assert GEO_RU_PROBE_LIMIT == 101, "Probe limit must be 101"
    assert GEO_RU_USE_WDQS_CACHE is False, "Final collection should use fresh WDQS calls by default"

    assert GEO_RU_TARGET_PLAN == {"L1": 5, "L2": 8, "L3": 12, "L4": 15, "L5": 15}, "Target plan must stay fixed at 5/8/12/15/15"
    assert sum(GEO_RU_TARGET_PLAN.values()) == 55, "Target plan must generate 55 examples"

    ru0, en0 = _gr_finalize_text("Назови 5 озёр России.", "Name 5 lakes in Russia.", total_gold=20, requested_count=5)
    assert ru0 == "Назови 5 озёр России." and en0 == "Name 5 lakes in Russia.", "Finalizer must not append extra list-all/list-any phrases"
    assert "Перечисли" not in ru0, "Query text must not contain redundant enumeration suffixes"
    assert "List any" not in en0 and "List all" not in en0, "Query text must not contain redundant enumeration suffixes"
    dirty_constraints = _gr_public_constraints({
        "answer_type": "lake",
        "country": COUNTRY_RUSSIA,
        "located_in_administrative_entity": _gr_entity("Q6563", "Krasnoyarsk Krai"),
        "area_min_km2": 100,
        "template_id": "x",
        "template_family": "x",
        "constraint_language": "en",
        "requires_ru_label": True,
        "requires_en_label": True,
        "requires_latin_script_en_label": True,
        "max_gold_allowed": 100,
        "excluded_answer_types": ["reservoir"],
        "population_unit": "people",
    })
    assert dirty_constraints == {
        "kind": "lake",
        "country": "Russia",
        "administrative_entity": "Krasnoyarsk Krai",
        "area_min_sqkm": 100,
    }, "Public constraints must use final JSONL style: semantic flat keys, no QIDs, no metadata"
    _gr_assert_public_constraints_clean(dirty_constraints)

    one_of_constraints = _gr_public_constraints({
        "answer_type": "river",
        "mouth_of_watercourse_one_of": [
            _gr_entity("Q166", "Black Sea"),
            _gr_entity("Q545", "Baltic Sea"),
        ],
        "length_min_km": 500,
    })
    assert one_of_constraints == {
        "kind": "river",
        "mouth_body_one_of": ["Black Sea", "Baltic Sea"],
        "length_min_km": 500,
    }
    _gr_assert_public_constraints_clean(one_of_constraints)


    max_constraints = _gr_public_constraints({
        "answer_type": "river",
        "country": COUNTRY_RUSSIA,
        "length_max_km": 1000,
    })
    assert max_constraints == {"kind": "river", "country": "Russia", "length_max_km": 1000}, "Public constraints must preserve max numeric criteria"
    range_constraints = _gr_public_constraints({
        "answer_type": "lake",
        "area_min_km2": 10,
        "area_max_km2": 100,
        "same_administrative_entity_has_lake_area_max_km2": 500,
    })
    assert range_constraints == {
        "kind": "lake",
        "area_min_sqkm": 10,
        "area_max_sqkm": 100,
        "same_administrative_entity_has_lake_area_max_sqkm": 500,
    }, "Public constraints must rename km² min/max keys to sqkm style"
    sample_bound = {"min": None, "max": 1000}
    assert "не более" in _gr_length_clause_ru(sample_bound), "Numeric text must support max / не более criteria"
    assert "at most" in _gr_length_clause_en(sample_bound), "English numeric text must support max criteria"
    old_mode = globals().get("_GEO_RU_FORCED_NUMERIC_MODE")
    try:
        globals()["_GEO_RU_FORCED_NUMERIC_MODE"] = "max"
        assert _gr_bound_mode(_gr_pick_bound(random.Random(1), [10, 20, 30], modes=("min", "max", "range"))) == "max"
        globals()["_GEO_RU_FORCED_NUMERIC_MODE"] = "range"
        assert _gr_bound_mode(_gr_pick_bound(random.Random(1), [10, 20, 30], modes=("min", "max", "range"))) == "range"
        globals()["_GEO_RU_FORCED_NUMERIC_MODE"] = "min"
        assert _gr_bound_mode(_gr_pick_bound(random.Random(1), [10, 20, 30], modes=("min", "max", "range"))) == "min"
    finally:
        globals()["_GEO_RU_FORCED_NUMERIC_MODE"] = old_mode

    assert set(GEO_RU_NUMERIC_MODE_PLAN["L1"][:3]) >= {"min", "range"}, "L1 numeric mode plan must include early min/range diversity"



_gr_run_sanity_checks()
assert len(set(GEO_RU_TEMPLATE_PLAN_BY_LEVEL["L1"][:3])) == 3, "L1 template plan must start with three different task families"
_disabled_templates = set(globals().get("GEO_RU_DISABLED_TEMPLATE_IDS", set()))
assert "geo_ru_admin_center_subject_population" in _disabled_templates
assert "geo_ru_admin_centers_subject_area" in _disabled_templates
for _lvl, _plan in GEO_RU_TEMPLATE_PLAN_BY_LEVEL.items():
    _overlap = _disabled_templates.intersection(set(_plan))
    assert not _overlap, f"Pure admin-centre templates must not be scheduled for {_lvl}: {_overlap}"
assert "forced_template_id" in generate_geo_ru_example.__annotations__ or True

# v16: protected-area prompts intentionally use broad "protected areas", not
# "protected natural areas", because Wikidata's protected-area class can include
# parks, arboretums, museum-reserves and similar protected sites.
_pa_consts = " ".join(str(x) for x in _gr_tpl_protected_areas_subject_capital.__code__.co_consts)
assert "охраняемые природные территории" not in _pa_consts
assert "охраняемые территории" in _pa_consts
assert _gr_record_key(type("Dummy", (), {"template_id": "t", "constraints": {"kind": "city"}, "query_text_en": "a", "complexity": "L3"})()) == _gr_record_key(type("Dummy", (), {"template_id": "t", "constraints": {"kind": "city"}, "query_text_en": "b", "complexity": "L4"})())
print("✅ geo_ru quality sanity checks passed: target plan 5/8/12/15/15, clean constraints, strict RU+EN labels, LIMIT 101, no reservoirs, min/max/range criteria, v15 fast exact-template probes + visible fallback plan + pre-WDQS diversity")


✅ geo_ru quality sanity checks passed: target plan 5/8/12/15/15, clean constraints, strict RU+EN labels, LIMIT 101, no reservoirs, min/max/range criteria, v15 fast exact-template probes + visible fallback plan + pre-WDQS diversity


## 9. Dataset generation runner

This cell creates `out_wikidata_benchmark/domain_outputs/geo_ru.jsonl` plus audit/checkpoint files. It is safe to interrupt: completed records are saved incrementally.


In [10]:
def _gr_append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _gr_write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


def _gr_record_numeric_mode(ex: BenchmarkExample) -> str:
    return _gr_constraint_numeric_mode(getattr(ex, "constraints", {}) or {})


def _gr_audit(records: Sequence[BenchmarkExample], skipped: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "domain": "geo_ru",
        "target_plan": GEO_RU_TARGET_PLAN,
        "generated": len(records),
        "counts_by_complexity": dict(Counter(x.complexity for x in records)),
        "counts_by_template_family": dict(Counter(x.template_family for x in records)),
        "counts_by_template_id": dict(Counter(x.template_id for x in records)),
        "numeric_mode_counts": dict(Counter(_gr_record_numeric_mode(x) for x in records)),
        "numeric_mode_counts_by_complexity": {lvl: dict(Counter(_gr_record_numeric_mode(x) for x in records if x.complexity == lvl)) for lvl in GEO_RU_TARGET_PLAN},
        "numeric_mode_plan": {k: list(v) for k, v in GEO_RU_NUMERIC_MODE_PLAN.items()},
        "template_plan": {k: list(v) for k, v in GEO_RU_TEMPLATE_PLAN_BY_LEVEL.items()},
        "fast_pre_wdqs_diversity": True,
        "v13_one_wdqs_probe_per_generator_step": True,
        "v16_no_cross_level_exact_duplicate_constraints": True,
        "v16_protected_area_ru_wording_broadened": True,
        "v17_all_russian_federal_subject_types": True,
        "v18_nature_heavy_template_plan": True,
        "v18_l2_admin_centres_removed_from_plan": True,
        "v19_pure_admin_centres_disabled": True,
        "v19_nature_heavy_no_admin_center_bias": True,
        "disabled_template_ids": sorted(list(globals().get("GEO_RU_DISABLED_TEMPLATE_IDS", set()))),
        "v18_new_templates": ["geo_ru_mountain_or_volcano_region_elevation", "geo_ru_protected_areas_region_inception"],
        "use_memory_wdqs_cache": bool(globals().get("GEO_RU_USE_MEMORY_WDQS_CACHE", True)),
        "gold_size_by_complexity": {
            lvl: {
                "min": min([len(x.gold_answer_qids) for x in records if x.complexity == lvl] or [0]),
                "max": max([len(x.gold_answer_qids) for x in records if x.complexity == lvl] or [0]),
                "avg": round(sum(len(x.gold_answer_qids) for x in records if x.complexity == lvl) / max(1, sum(1 for x in records if x.complexity == lvl)), 2),
            }
            for lvl in GEO_RU_TARGET_PLAN
        },
        "skipped_count": len(skipped),
        "skipped_preview": list(skipped)[-50:],
        "seed": GEO_RU_SEED,
        "output_path": str(GEO_RU_OUTPUT_PATH),
    }



from contextlib import contextmanager

@contextmanager
def _gr_wdqs_fail_fast_context():
    """Temporarily shorten WDQS timeout/retry chain for failed candidate probes."""
    if not bool(globals().get("GEO_RU_WDQS_FAIL_FAST", True)):
        yield
        return
    wd_obj = globals().get("wd")
    if wd_obj is None:
        yield
        return
    old_timeout = getattr(wd_obj, "timeout", None)
    old_retries = getattr(wd_obj, "max_retries", None)
    try:
        try:
            wd_obj.timeout = int(globals().get("GEO_RU_WDQS_FAST_TIMEOUT_SECONDS", 12))
            wd_obj.max_retries = int(globals().get("GEO_RU_WDQS_FAST_MAX_RETRIES", 1))
        except Exception:
            pass
        yield
    finally:
        try:
            if old_timeout is not None:
                wd_obj.timeout = old_timeout
            if old_retries is not None:
                wd_obj.max_retries = old_retries
        except Exception:
            pass

def _gr_slot_try_plan(level: str, ok_index: int) -> List[Tuple[str, Optional[str]]]:
    """Return a small visible fallback plan for one accepted slot.

    This is pre-WDQS and therefore cheap.  It preserves diversity while avoiding
    the old hidden "try every template" inner loop.
    """
    desired_template_id = _gr_desired_template_id(level, ok_index)
    desired_mode = _gr_desired_numeric_mode(level, ok_index)

    template_ids: List[str] = []
    if desired_template_id:
        template_ids.append(desired_template_id)
    for tid in GEO_RU_TEMPLATE_PLAN_BY_LEVEL.get(level, ()):
        if tid not in template_ids:
            template_ids.append(tid)
    for fn in GEO_RU_TEMPLATE_FNS.get(level, []):
        for tid, f in GEO_RU_TEMPLATE_ID_TO_FN.items():
            if f is fn and tid not in template_ids:
                template_ids.append(tid)
                break
    disabled = set(globals().get("GEO_RU_DISABLED_TEMPLATE_IDS", set()))
    template_ids = [tid for tid in template_ids if tid not in disabled]
    template_ids = template_ids[:4]

    if desired_mode in {"min", "max", "range"}:
        modes: List[Optional[str]] = [desired_mode] + [m for m in ("range", "max", "min") if m != desired_mode]
    else:
        modes = [None]

    plan: List[Tuple[str, Optional[str]]] = []
    for i, tid in enumerate(template_ids):
        mode = modes[min(i, len(modes) - 1)] if modes else None
        plan.append((tid, mode))
    return plan


def generate_geo_ru_dataset(
    target_plan: Optional[Dict[str, int]] = None,
    output_path: Path = GEO_RU_OUTPUT_PATH,
    audit_path: Path = GEO_RU_AUDIT_PATH,
    overwrite: bool = OVERWRITE_GEO_RU_OUTPUT,
    seed: int = GEO_RU_SEED,
    max_attempts_per_level: int = 180,
) -> List[BenchmarkExample]:
    target_plan = dict(target_plan or GEO_RU_TARGET_PLAN)
    rng = random.Random(seed)
    records: List[BenchmarkExample] = []
    skipped: List[Dict[str, Any]] = []
    seen: set[Tuple[str, str, str]] = set()

    old_fast_ctx = globals().get("_GEO_RU_FAST_DIVERSITY_CONTEXT")
    globals()["_GEO_RU_FAST_DIVERSITY_CONTEXT"] = {
        "region_orders": {},
        "region_cursors": defaultdict(int),
    }
    globals().setdefault("_GEO_RU_MEMORY_WDQS_ROWS_CACHE", {}).clear()

    if overwrite and output_path.exists():
        output_path.unlink()

    total_target = sum(target_plan.values())
    overall_bar = tqdm(total=total_target, desc="geo_ru total") if tqdm is not None else None
    idx = 1

    try:
        with _gr_wdqs_fail_fast_context():
            for level, target in target_plan.items():
                ok = 0
                attempts = 0
                level_bar = tqdm(total=target, desc=f"geo_ru {level}") if tqdm is not None else None
                while ok < target and attempts < max_attempts_per_level:
                    accepted: Optional[BenchmarkExample] = None
                    for template_id, mode in _gr_slot_try_plan(level, ok):
                        attempts += 1
                        ex = generate_geo_ru_example(
                            level,
                            idx,
                            rng=rng,
                            max_attempts=GEO_RU_GENERATOR_MAX_ATTEMPTS,
                            forced_numeric_mode=mode,
                            forced_template_id=template_id,
                        )
                        if not ex.gold_answer_qids:
                            skipped.append({"complexity": level, "reason": "empty_or_failed_generation", "template_id": template_id, "mode": mode})
                            continue
                        key = _gr_record_key(ex)
                        if key in seen:
                            skipped.append({"complexity": level, "reason": "duplicate", "template_id": ex.template_id, "mode": _gr_record_numeric_mode(ex)})
                            continue
                        accepted = ex
                        break

                    if accepted is None:
                        continue

                    seen.add(_gr_record_key(accepted))
                    records.append(accepted)
                    _gr_append_jsonl(output_path, asdict(accepted))
                    idx += 1
                    ok += 1
                    if level_bar is not None:
                        level_bar.update(1)
                        level_bar.set_postfix({"attempts": attempts, "gold": len(accepted.gold_answer_qids), "mode": _gr_record_numeric_mode(accepted), "tpl": accepted.template_id})
                    if overall_bar is not None:
                        overall_bar.update(1)
                        overall_bar.set_postfix({"level": level, "gold": len(accepted.gold_answer_qids)})
                    _gr_write_json(GEO_RU_CHECKPOINT_PATH, {"last_record": asdict(accepted), "audit": _gr_audit(records, skipped)})

                if level_bar is not None:
                    level_bar.close()
                if ok < target:
                    print(f"[WARN] geo_ru {level}: generated {ok}/{target} after {attempts} visible template attempts")
                else:
                    print(f"OK geo_ru {level}: generated {ok}/{target} after {attempts} visible template attempts")
    finally:
        if overall_bar is not None:
            overall_bar.close()
        audit = _gr_audit(records, skipped)
        _gr_write_json(audit_path, audit)
        print("saved:", output_path.resolve())
        print("audit:", audit_path.resolve())
        print("records:", len(records))
        print("counts:", dict(Counter(x.complexity for x in records)))
        print("families:", dict(Counter(x.template_family for x in records)))
        print("skipped:", len(skipped))

        if old_fast_ctx is None:
            globals().pop("_GEO_RU_FAST_DIVERSITY_CONTEXT", None)
        else:
            globals()["_GEO_RU_FAST_DIVERSITY_CONTEXT"] = old_fast_ctx
    return records


if RUN_GEO_RU_GENERATION:
    GEO_RU_RECORDS = generate_geo_ru_dataset()
else:
    print("RUN_GEO_RU_GENERATION is False; generator has been registered but dataset generation did not run.")


geo_ru L1: 100%|██████████| 5/5 [12:37<00:00, 151.45s/it, attempts=26, gold=7, mode=range_or_mixed, tpl=geo_ru_city_region_population]


OK geo_ru L1: generated 5/5 after 26 visible template attempts


geo_ru L2: 100%|██████████| 8/8 [21:40<00:00, 162.52s/it, attempts=62, gold=6, mode=min, tpl=geo_ru_city_region_population]


OK geo_ru L2: generated 8/8 after 62 visible template attempts


geo_ru L3: 100%|██████████| 12/12 [06:50<00:00, 34.20s/it, attempts=22, gold=34, mode=max, tpl=geo_ru_mountain_or_volcano_region_elevation]


OK geo_ru L3: generated 12/12 after 22 visible template attempts


geo_ru L4: 100%|██████████| 15/15 [13:34<00:00, 54.29s/it, attempts=51, gold=3, mode=range_or_mixed, tpl=geo_ru_rivers_mouth_to_sea_length]


OK geo_ru L4: generated 15/15 after 51 visible template attempts


geo_ru total: 100%|██████████| 55/55 [1:15:18<00:00, 82.15s/it, level=L5, gold=13]

OK geo_ru L5: generated 15/15 after 70 visible template attempts
saved: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/geo_ru.jsonl
audit: /Users/matvey/Desktop/multihop benchmark/multihop_benchmark_modular/out_wikidata_benchmark/domain_outputs/geo_ru_generation_audit.json
records: 55
counts: {'L1': 5, 'L2': 8, 'L3': 12, 'L4': 15, 'L5': 15}
families: {'lakes': 3, 'cities_admin': 7, 'rivers': 3, 'mountains_volcanoes': 16, 'protected_areas': 5, 'rivers_hydrography': 7, 'protected_areas_multihop': 1, 'protected_areas_mountains': 11, 'cities_hydrography': 2}
skipped: 176
